In [ ]:
# ============================================================================
# CELL 1: State Management & Utilities
# ============================================================================
# CHANGES FROM PREVIOUS:
# 1. Dual BASE_PATH detection — tries both known paths, uses whichever exists
# 2. Added all new file path constants (bag, timeline, item knowledge, etc.)
# 3. Expanded DEFAULT_BATTLE_DATA to match AI agent exactly (all fields)
# 4. Added DEFAULT_PARTY_DATA, DEFAULT_MENU_DATA, DEFAULT_BAG_DATA
# 5. Full parse_battle_data() — all fields including moves, pp, stat stages
# 6. Added parse_party_data(), parse_menu_data(), parse_bag_data()
# 7. Updated parse_game_state_data() — returns 9 values
# 8. Updated read_game_state() — returns 10 values matching AI agent
# ============================================================================

from pathlib import Path
import json
import numpy as np
import time
from collections import deque

# === DUAL BASE_PATH DETECTION ===
# Tries both known paths — uses whichever exists on this machine.
# This lets the same notebook run on either the teaching or AI machine.
_CANDIDATE_PATHS = [
    Path("C:/Users/HP/Documents/cogai/"),
    Path("C:/Users/natmaw/Documents/Boston Stuff/CS 5100 Foundations of AI/PokeAI/"),
]

BASE_PATH = None
for _p in _CANDIDATE_PATHS:
    if _p.exists():
        BASE_PATH = _p
        break

if BASE_PATH is None:
    # Fallback: use first candidate and let errors surface naturally
    BASE_PATH = _CANDIDATE_PATHS[0]
    print(f"⚠️ WARNING: No valid base path found. Defaulting to {BASE_PATH}")
else:
    print(f"📂 BASE_PATH: {BASE_PATH}")

# === FILE PATHS ===
ACTION_FILE = BASE_PATH / "action.json"
STATE_FILE = BASE_PATH / "game_state.json"
INPUT_FILE = BASE_PATH / "input_cache.txt"

# Teaching output files (produced by this code, consumed by AI agent)
TAUGHT_TRANSITIONS_FILE = BASE_PATH / "taught_transitions.json"
TAUGHT_BATTLE_TRANSITIONS_FILE = BASE_PATH / "taught_battle_transitions.json"
TAUGHT_BAG_TRANSITIONS_FILE = BASE_PATH / "taught_bag_transitions.json"
TAUGHT_NAV_TARGETS_FILE = BASE_PATH / "taught_nav_targets.json"
MODEL_CHECKPOINT_FILE = BASE_PATH / "taught_model_checkpoint.json"
EXPLORATION_MEMORY_FILE = BASE_PATH / "taught_exploration_memory.json"
EVENT_TIMELINE_FILE = BASE_PATH / "event_timeline.json"

# Aliases for backward compat
MODEL_FILE = MODEL_CHECKPOINT_FILE
TAUGHT_EXPLORATION_FILE = EXPLORATION_MEMORY_FILE

# AI agent files (read-only references, if they exist)
AI_MODEL_CHECKPOINT_FILE = BASE_PATH / "model_checkpoint.json"
AI_EXPLORATION_MEMORY_FILE = BASE_PATH / "exploration_memory.json"
ROSTER_FILE = BASE_PATH / "roster.json"
MOVE_KNOWLEDGE_FILE = BASE_PATH / "move_knowledge.json"
ITEM_KNOWLEDGE_FILE = BASE_PATH / "item_knowledge.json"

# === MARKOV SIMILARITY WEIGHTS (consistent with AI agent) ===
MARKOV_IMMEDIATE_WEIGHT = 0.5
MARKOV_SEQUENTIAL_WEIGHT = 0.3
MARKOV_PARTIAL_WEIGHT = 0.2
MARKOV_FAMILIARITY_THRESHOLD = 0.6

MARKOV_SEQ_FULL_WEIGHT = 1.0
MARKOV_SEQ_MEDIUM_WEIGHT = 0.6
MARKOV_SEQ_SHORT_WEIGHT = 0.3

MARKOV_POS_EXACT_BONUS = 0.35
MARKOV_POS_NEAR_BONUS = 0.25
MARKOV_POS_FAR_BONUS = 0.1
MARKOV_POS_MAX_DIST = 5

EXPECTED_STATE_DIM = 6
PALETTE_DIM = 768
TILE_DIM = 600
LEARNING_STATE_DIM = 8 + TILE_DIM + PALETTE_DIM  # 1376

# Action code mapping (from Lua short codes)
ACTION_MAP = {
    'U': 'UP', 'D': 'DOWN', 'L': 'LEFT', 'R': 'RIGHT',
    'A': 'A', 'B': 'B', 'S': 'Start', 'E': 'Select'
}

# === DEFAULT BATTLE DATA (matches AI agent exactly) ===
DEFAULT_BATTLE_DATA = {
    'battle_cursor': -1,
    'move_cursor': -1,
    'party_cursor': -1,
    'player_species': -1,
    'enemy_species': -1,
    'player_hp': -1,
    'player_max_hp': -1,
    'enemy_hp': -1,
    'enemy_max_hp': -1,
    'player_level': -1,
    'enemy_level': -1,
    'player_status': 0,
    'enemy_status': 0,
    'battle_type': 0,
    'move0': -1, 'move1': -1, 'move2': -1, 'move3': -1,
    'pp0': -1, 'pp1': -1, 'pp2': -1, 'pp3': -1,
    'player_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
    'enemy_move0': -1, 'enemy_move1': -1, 'enemy_move2': -1, 'enemy_move3': -1,
    'enemy_pp0': -1, 'enemy_pp1': -1, 'enemy_pp2': -1, 'enemy_pp3': -1,
    'enemy_stat_stages': [-1, -1, -1, -1, -1, -1, -1],
}

# === DEFAULT PARTY DATA (matches AI agent) ===
DEFAULT_PARTY_SLOT = {
    'level': 0, 'hp': 0, 'max_hp': 0,
    'atk': 0, 'def': 0, 'spd': 0, 'spatk': 0, 'spdef': 0,
    'status': 0
}

DEFAULT_PARTY_DATA = {
    'count': 0,
    'slots': []
}

# === DEFAULT MENU DATA (from "mu" field — matches AI agent) ===
DEFAULT_MENU_DATA = {
    'mc': -1,
    'mm': -1,
    'pc': -1,
    'sc': -1,
}

# === DEFAULT BAG DATA (from "bg" field — matches AI agent) ===
DEFAULT_BAG_DATA = {
    'pocket': -1,
    'cursor': -1,
    'active': 0,
    'items': [],
}


def normalize_game_state(raw_state):
    if len(raw_state) < 6:
        raw_state = list(raw_state) + [0] * (6 - len(raw_state))
    normalized = np.array(raw_state, dtype=float)
    normalized[0] = raw_state[0] / 255.0
    normalized[1] = raw_state[1] / 255.0
    normalized[2] = np.clip(raw_state[2], 0, 255)
    normalized[3] = 1.0 if raw_state[3] > 0 else 0.0
    normalized[4] = 1.0 if raw_state[4] > 0 else 0.0
    normalized[5] = int(raw_state[5]) % 4
    return normalized


def compute_derived_features(current, prev):
    if prev is None:
        return np.zeros(8)
    vel_x = current[0] - prev[0]
    vel_y = current[1] - prev[1]
    map_changed = 1.0 if abs(current[2] - prev[2]) > 0.5 else 0.0
    battle_started = 1.0 if current[3] > prev[3] else 0.0
    battle_ended = 1.0 if current[3] < prev[3] else 0.0
    menu_opened = 1.0 if current[4] > prev[4] else 0.0
    menu_closed = 1.0 if current[4] < prev[4] else 0.0
    direction_changed = 1.0 if current[5] != prev[5] else 0.0
    return np.array([vel_x, vel_y, map_changed, battle_started, battle_ended,
                     menu_opened, menu_closed, direction_changed])


def build_learning_state(derived, palette, tiles, in_battle):
    """Build learning state — matches AI agent's version exactly."""
    if len(derived) != 8:
        derived = np.zeros(8)
    if len(tiles) != TILE_DIM:
        tiles = np.zeros(TILE_DIM)
    if len(palette) != PALETTE_DIM:
        palette = np.zeros(PALETTE_DIM)

    if in_battle > 0.5:
        state = np.concatenate([derived, palette])
    else:
        state = np.concatenate([derived, tiles, palette])
    noise = np.random.randn(len(state)) * 0.0001
    return state + noise


def _pad_or_trim(arr, target_dim):
    """Dimension safety helper."""
    if arr.shape[0] < target_dim:
        return np.pad(arr, (0, target_dim - arr.shape[0]))
    elif arr.shape[0] > target_dim:
        return arr[:target_dim]
    return arr


def parse_battle_data(data):
    """
    Parse battle data from the 'b' field in game_state.json.
    Matches AI agent's parse_battle_data() exactly.

    Lua writes short keys: bc, mc, ps, es, ph, pm, eh, em, pl, el,
    pst, est, bt, m0-m3, pp0-pp3, pss[7], em0-em3, epp0-epp3, ess[7], pc.

    Returns dict with full key names matching DEFAULT_BATTLE_DATA.
    """
    b = data.get('b')
    if b is None:
        return DEFAULT_BATTLE_DATA.copy()

    pss = b.get('pss')
    if not isinstance(pss, list) or len(pss) != 7:
        pss = [-1, -1, -1, -1, -1, -1, -1]
    else:
        pss = list(pss)

    ess = b.get('ess')
    if not isinstance(ess, list) or len(ess) != 7:
        ess = [-1, -1, -1, -1, -1, -1, -1]
    else:
        ess = list(ess)

    return {
        'battle_cursor': b.get('bc', -1),
        'move_cursor':   b.get('mc', -1),
        'party_cursor':  b.get('pc', -1),
        'player_species': b.get('ps', -1),
        'enemy_species':  b.get('es', -1),
        'player_hp':     b.get('ph', -1),
        'player_max_hp': b.get('pm', -1),
        'enemy_hp':      b.get('eh', -1),
        'enemy_max_hp':  b.get('em', -1),
        'player_level':  b.get('pl', -1),
        'enemy_level':   b.get('el', -1),
        'player_status': b.get('pst', 0),
        'enemy_status':  b.get('est', 0),
        'battle_type':   b.get('bt', 0),
        'move0': b.get('m0', -1),
        'move1': b.get('m1', -1),
        'move2': b.get('m2', -1),
        'move3': b.get('m3', -1),
        'pp0': b.get('pp0', -1),
        'pp1': b.get('pp1', -1),
        'pp2': b.get('pp2', -1),
        'pp3': b.get('pp3', -1),
        'player_stat_stages': pss,
        'enemy_move0': b.get('em0', -1),
        'enemy_move1': b.get('em1', -1),
        'enemy_move2': b.get('em2', -1),
        'enemy_move3': b.get('em3', -1),
        'enemy_pp0': b.get('epp0', -1),
        'enemy_pp1': b.get('epp1', -1),
        'enemy_pp2': b.get('epp2', -1),
        'enemy_pp3': b.get('epp3', -1),
        'enemy_stat_stages': ess,
    }


def parse_party_data(data):
    """
    Parse party data from the 'pa' field in game_state.json.
    Matches AI agent's parse_party_data() exactly.

    Lua writes: pa.c = count, pa.s = [{l, h, m, a, d, sp, sa, sd, st}, ...]
    """
    pa = data.get('pa')
    if pa is None:
        return DEFAULT_PARTY_DATA.copy()

    count = pa.get('c', 0)
    raw_slots = pa.get('s', [])

    slots = []
    for s in raw_slots:
        if not isinstance(s, dict):
            continue
        slots.append({
            'level':  s.get('l', 0),
            'hp':     s.get('h', 0),
            'max_hp': s.get('m', 0),
            'atk':    s.get('a', 0),
            'def':    s.get('d', 0),
            'spd':    s.get('sp', 0),
            'spatk':  s.get('sa', 0),
            'spdef':  s.get('sd', 0),
            'status': s.get('st', 0),
        })

    return {
        'count': count,
        'slots': slots
    }


def parse_menu_data(data):
    """
    Parse menu data from the 'mu' field in game_state.json.
    Matches AI agent's parse_menu_data() exactly.

    Lua writes: mu.mc, mu.mm, mu.pc, mu.sc
    """
    mu = data.get('mu')
    if mu is None:
        return DEFAULT_MENU_DATA.copy()

    return {
        'mc': mu.get('mc', -1),
        'mm': mu.get('mm', -1),
        'pc': mu.get('pc', -1),
        'sc': mu.get('sc', -1),
    }


def parse_bag_data(data):
    """
    Parse bag data from the 'bg' field in game_state.json.
    Matches AI agent's parse_bag_data() exactly.

    Lua writes: bg.pk, bg.bc, bg.a, bg.it = [{id, q}, ...]
    """
    bg = data.get('bg')
    if bg is None:
        return DEFAULT_BAG_DATA.copy()

    items = []
    raw_items = bg.get('it', [])
    if isinstance(raw_items, list):
        for item in raw_items:
            if isinstance(item, dict) and 'id' in item:
                items.append({
                    'id': item.get('id', 0),
                    'q': item.get('q', 0),
                })

    return {
        'pocket': bg.get('pk', -1),
        'cursor': bg.get('bc', -1),
        'active': bg.get('a', 0),
        'items': items,
    }


def parse_game_state_data(data):
    """Parse game state dict handling both long and short key formats.

    Returns: (raw, palette_raw, tiles_raw, dead, battle_data, party_data,
              game_state_raw, menu_data, bag_data)
    """
    raw = data.get("state") or data.get("s") or []
    palette_raw = data.get("palette") or data.get("p") or []
    tiles_raw = data.get("tiles") or data.get("t") or []
    dead = bool(data.get("dead", False))
    battle_data = parse_battle_data(data)
    party_data = parse_party_data(data)
    game_state_raw = data.get("gs", 0)
    menu_data = parse_menu_data(data)
    bag_data = parse_bag_data(data)
    return raw, palette_raw, tiles_raw, dead, battle_data, party_data, game_state_raw, menu_data, bag_data


def read_game_state(max_retries=3):
    """
    Read game state from JSON file.

    Returns: (context_state, palette_state, tile_state, dead,
              raw_position, battle_data, party_data,
              game_state_raw, menu_data, bag_data)

    Matches AI agent's read_game_state() signature exactly.
    """
    if not STATE_FILE.exists():
        return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy())

    for attempt in range(max_retries):
        try:
            with open(STATE_FILE, "r") as f:
                data = json.loads(f.read())

            (raw, palette_raw, tiles_raw, dead, battle_data, party_data,
             game_state_raw, menu_data, bag_data) = parse_game_state_data(data)

            raw_x = int(raw[0]) if len(raw) > 0 else 0
            raw_y = int(raw[1]) if len(raw) > 1 else 0
            raw_position = (raw_x, raw_y)

            context_state = normalize_game_state(np.array(raw, dtype=float))
            palette_state = np.array(palette_raw, dtype=float) if palette_raw else np.zeros(PALETTE_DIM)
            tile_state = np.array(tiles_raw, dtype=float) if tiles_raw else np.zeros(TILE_DIM)

            context_state = _pad_or_trim(context_state, EXPECTED_STATE_DIM)
            palette_state = _pad_or_trim(palette_state, PALETTE_DIM)
            tile_state = _pad_or_trim(tile_state, TILE_DIM)

            return (context_state, palette_state, tile_state, dead, raw_position,
                    battle_data, party_data, game_state_raw, menu_data, bag_data)

        except (json.JSONDecodeError, ValueError):
            if attempt < max_retries - 1:
                time.sleep(0.001)
                continue
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy())
        except Exception:
            return (np.zeros(EXPECTED_STATE_DIM), np.zeros(PALETTE_DIM), np.zeros(TILE_DIM),
                    False, (0, 0), DEFAULT_BATTLE_DATA.copy(), DEFAULT_PARTY_DATA.copy(),
                    0, DEFAULT_MENU_DATA.copy(), DEFAULT_BAG_DATA.copy())


def write_action(action_name):
    if action_name:
        action_name = action_name.upper()
    try:
        with open(ACTION_FILE, "w") as f:
            json.dump({"action": action_name}, f)
            f.flush()
    except Exception as e:
        print(f"[ERROR] Failed to write action: {e}")

In [ ]:
# ============================================================================
# CELL 2: Perceptron Classes
# ============================================================================
# CHANGES FROM PREVIOUS:
# 1. Added self.cluster_activations = deque(maxlen=50) — longer history for
#    entity clustering, matching AI agent's Perceptron exactly
# 2. Entity predict() now appends to cluster_activations — enables clustering
#    to compare activation patterns across entities during teaching
# 3. All other code unchanged
# ============================================================================

class Perceptron:
    def __init__(self, kind, action=None, group=None, entity_type=None):
        self.kind = kind
        self.action = action
        self.group = group
        self.entity_type = entity_type
        
        self.utility = 1.0
        self.weights = None
        
        self.eligibility_fast = 0.0
        self.eligibility_slow = 0.0
        
        self.familiarity = 0.0
        self.activation_history = deque(maxlen=10)
        self.cluster_activations = deque(maxlen=50)  # NEW: for entity clustering
        
        self.learning_rate = 0.01
        self.prediction_errors = deque(maxlen=50)

    def ensure_weights(self, dim):
        """Initialize or resize weights to match dimension."""
        if self.weights is None:
            self.weights = np.random.randn(dim) * 0.001
        elif len(self.weights) != dim:
            old_weights = self.weights
            self.weights = np.random.randn(dim) * 0.001
            min_len = min(len(old_weights), dim)
            self.weights[:min_len] = old_weights[:min_len]

    def predict(self, state):
        self.ensure_weights(len(state))
        
        # Handle dimension mismatch
        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            raw_activation = np.dot(self.weights[:min_dim], state[:min_dim])
        else:
            raw_activation = np.dot(self.weights, state)
        
        if self.kind == "entity":
            novelty_factor = 1.0 / (1.0 + np.sqrt(self.familiarity * 0.5))
            decayed_activation = raw_activation * novelty_factor
            self.activation_history.append(abs(raw_activation))
            self.cluster_activations.append(abs(raw_activation))  # NEW: for clustering
            return decayed_activation
        else:
            return raw_activation

    def adapt_learning_rate(self):
        if len(self.prediction_errors) >= 50:
            avg_error = np.mean(self.prediction_errors)
            if avg_error < 0.1:
                self.learning_rate = max(0.001, self.learning_rate * 0.99)
            elif avg_error > 0.5:
                self.learning_rate = min(0.05, self.learning_rate * 1.01)

    def update(self, state, error, gamma_fast=0.5, gamma_slow=0.95, stagnation=0.0):
        self.ensure_weights(len(state))
        
        # Handle dimension mismatch
        if len(self.weights) != len(state):
            min_dim = min(len(self.weights), len(state))
            state = state[:min_dim]
            self.weights = self.weights[:min_dim]
        
        self.eligibility_fast = gamma_fast * self.eligibility_fast + 1.0
        self.eligibility_slow = gamma_slow * self.eligibility_slow + 1.0
        
        self.adapt_learning_rate()
        
        fast_update = 0.7 * self.learning_rate * error * state * self.eligibility_fast
        slow_update = 0.3 * self.learning_rate * error * state * self.eligibility_slow
        self.weights += fast_update + slow_update

        if self.kind == "action":
            if error > 0.01:
                if stagnation > 0.5:
                    self.utility *= 0.97
                elif error > 0.2:
                    self.utility = min(self.utility * 1.02, 2.0)
                else:
                    self.utility *= 0.995
            
            if self.group == "move":
                self.utility = np.clip(self.utility, 0.1, 2.0)
            else:
                self.utility = np.clip(self.utility, 0.01, 2.0)
        
        if self.kind == "entity" and len(self.activation_history) > 0:
            recent_avg = np.mean(self.activation_history)
            if recent_avg > 0.1:
                self.familiarity += 0.03
        
        if self.kind == "entity":
            prediction = self.predict(state)
            self.prediction_errors.append(abs(prediction - error))


class ControlSwapPerceptron(Perceptron):
    def __init__(self):
        super().__init__(kind="control_swap")
        self.swap_history = deque(maxlen=100)
        self.confidence = 0.0
        
    def should_swap(self, state, movement_stagnation):
        if self.weights is None:
            return False, 0.0
        self.ensure_weights(len(state))
        swap_score = np.dot(self.weights, state)
        stagnation_factor = np.tanh(movement_stagnation / 5.0)
        combined_score = swap_score * 0.7 + stagnation_factor * 0.3
        return combined_score > 0.5, abs(combined_score)
    
    def record_swap_outcome(self, state, swapped, novelty_gained):
        self.swap_history.append((swapped, novelty_gained))
        if len(self.swap_history) >= 20:
            recent = list(self.swap_history)[-20:]
            successful = sum(1 for swap, nov in recent if swap and nov > 0.2)
            self.confidence = successful / 20.0

In [ ]:
# ============================================================================
# CELL 3 PART 1: Brain Class — Initialization + New Data Management Methods
# ============================================================================
# CHANGES FROM PREVIOUS:
# 1. NEW state variables: party_data, prev_party_data, menu_data, prev_menu_data,
#    bag_data, prev_bag_data, game_state_raw, human_action
# 2. NEW: update_party_data(), update_menu_data(), update_bag_data()
# 3. NEW: Party context helpers — get_party_hp_ratios(), get_party_status_flags(),
#    get_lowest_hp_ratio(), has_status_condition_in_party(), get_party_snapshot(),
#    get_party_context_for_bag()
# 4. NEW: get_item_at_cursor() — item ID at current bag cursor
# 5. NEW: set_human_action() — tracks human's actual button press each frame
# 6. NEW: Bag session recording — start_bag_session(), record_bag_frame(),
#    end_bag_session(), get_accumulated_bag_data()
# 7. NEW: recent_actions_buffer expanded to maxlen=15 for bag Markov matching
# 8. NEW: Entity spawning/clustering state variables matching AI agent
# 9. All existing state variables preserved
# ============================================================================

import gc
import random

class Brain:
    def __init__(self):
        self.perceptrons = []
        self.prev_learning_states = deque(maxlen=50)
        self.prev_context_states = deque(maxlen=10)
        self.last_positions = deque(maxlen=30)
        self.action_history = deque(maxlen=100)
        
        self.control_mode = "move"
        self.timestep = 0
        self.last_action = None
        self.last_direction = 0
        
        # === HUMAN ACTION TRACKING (NEW) ===
        # The human's actual button press this frame, read from Lua input.
        # This is set via set_human_action() each frame before learn() is called,
        # ensuring brain.last_action reflects what the human actually did.
        self.human_action = None
        
        self.MOVE_UTILITY_FLOOR = 0.05
        self.INTERACT_UTILITY_FLOOR = 0.15
        
        # === EXPLORATION MEMORY ===
        self.EXPLORATION_MEMORY_FILE = BASE_PATH / "taught_exploration_memory.json"
        self.exploration_memory = {}
        self.current_map_id = None
        self.SAVE_INTERVAL = 100
        self.MAX_MAPS_IN_MEMORY = 20
        
        self.DIRECTION_NAMES = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
        self.DIRECTION_TO_INT = {"DOWN": 0, "UP": 1, "LEFT": 2, "RIGHT": 3}
        self.INT_TO_ACTION = {0: "DOWN", 1: "UP", 2: "LEFT", 3: "RIGHT"}
        self.DIRECTION_DELTAS_INT = {0: (0, 1), 1: (0, -1), 2: (-1, 0), 3: (1, 0)}
        self.ACTION_DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
        self.DELTA_TO_DIRECTION = {(0, 1): 0, (0, -1): 1, (-1, 0): 2, (1, 0): 3}
        
        self.load_exploration_memory()
        
        # === MARKOV TRANSITION SYSTEM ===
        self.taught_transitions = []
        self.taught_batches = []
        self.taught_metadata = {}
        self.markov_enabled = True
        self.markov_action_count = 0
        self.curiosity_action_count = 0
        self.last_markov_score = 0.0
        self.last_markov_action = None
        self.recent_actions_buffer = deque(maxlen=15)  # Expanded from 8 for bag Markov
        
        # === TAUGHT MODEL REFERENCE ===
        self.taught_reference = {'utilities': {}, 'weights': {}, 'loaded': False}
        
        # === BLEND SYSTEM ===
        self.blend_tier = 0
        self.last_blend_timestep = 0
        self.BLEND_COOLDOWN = 50
        self.blend_count = 0
        self.BLEND_RATIOS = {1: (0.80, 0.20), 2: (0.60, 0.40), 3: (0.40, 0.60)}
        self.BLEND_TIER_TRIGGERS = {
            1: {'pattern_repeats': 3, 'pos_stagnation': 8, 'consecutive': 12},
            2: {'pattern_repeats': 6, 'pos_stagnation': 15, 'consecutive': 15},
            3: {'pattern_repeats': 10, 'state_stagnation_mult': 2.0}
        }
        
        # === BATTLE DATA FROM MEMORY ADDRESSES (EXPANDED — matches AI agent) ===
        self.battle_data = DEFAULT_BATTLE_DATA.copy()
        self.prev_battle_data = DEFAULT_BATTLE_DATA.copy()
        self.battle_data_buffer = []
        self.in_battle_last_frame = False
        self.current_battle_id = 0
        
        # === PARTY DATA (from "pa" field — NEW) ===
        self.party_data = DEFAULT_PARTY_DATA.copy()
        self.prev_party_data = DEFAULT_PARTY_DATA.copy()
        
        # === MENU DATA (from "mu" field — NEW) ===
        self.menu_data = DEFAULT_MENU_DATA.copy()
        self.prev_menu_data = DEFAULT_MENU_DATA.copy()
        
        # === BAG DATA (from "bg" field — NEW) ===
        self.bag_data = DEFAULT_BAG_DATA.copy()
        self.prev_bag_data = DEFAULT_BAG_DATA.copy()
        
        # === RAW GAME STATE (from "gs" field — NEW) ===
        self.game_state_raw = 0
        self.prev_game_state_raw = 0
        
        # === BAG SESSION RECORDING (NEW) ===
        # Accumulates bag frames during live recording for taught_bag_transitions.json
        self.bag_session_active = False
        self.bag_session_context = "none"  # "overworld" or "battle"
        self.bag_session_start_timestep = 0
        self.bag_session_frame_count = 0
        self.bag_session_last_action = None
        self.bag_session_items_used = []
        self.bag_session_pockets_visited = set()
        
        # Current session frames (cleared each session)
        self.bag_session_frames = []
        
        # Accumulated across all sessions (persisted to file)
        self.bag_sessions_accumulated = []
        self.bag_sessions_metadata = {
            'total_bag_frames': 0,
            'bag_sessions_recorded': 0,
            'items_used': set(),
            'pockets_visited': set(),
        }
        
        # === ACTION EXECUTION ===
        self.pending_action = None
        self.pending_action_frames = 0
        self.ACTION_CONFIRM_FRAMES = 3
        self.last_confirmed_action = None
        
        # === TILE INTERACTION ===
        self.INTERACTION_VERIFY_FRAMES = 8
        self.MIN_SUCCESS_RATE_THRESHOLD = 0.1
        self.pending_interaction_verify = None
        self.interaction_verify_countdown = 0
        
        # === MENU TRAP ===
        self.menu_trap_frames = 0
        self.menu_trap_b_boost = 1.0
        self.menu_trap_position = None
        self.B_BOOST_INCREMENT = 0.15
        self.B_BOOST_MAX = 3.0
        self.MENU_TRAP_THRESHOLD = 5
        self.original_b_utility = None
        
        # === MODE SWAPPING ===
        self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD = 15
        self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD = 25
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.THRESHOLD_INCREMENT = 15
        self.MAX_THRESHOLD = 150
        self.frames_in_current_mode = 0
        self.swap_chain_count = 0
        self.position_at_mode_swap = None
        self.last_map_id = None
        self.last_battle_state = None
        
        # === UNPRODUCTIVE SWAP ===
        self.UNPRODUCTIVE_SWAP_THRESHOLD = 3
        self.unproductive_swap_count = 0
        
        # === STATE STAGNATION ===
        self.STATE_STAGNATION_THRESHOLD = 20
        self.state_stagnation_count = 0
        self.last_context_state_hash = None
        self.stagnation_initiator_action = None
        
        # === BOTH MODE ===
        self.BOTH_MODE_STAGNATION_THRESHOLD = 35
        self.BOTH_MODE_SWAP_THRESHOLD = 5
        self.last_direction_for_progress = None
        self.direction_change_counts_as_progress = True
        
        # === NOVELTY WEIGHTS ===
        self.UNVISITED_TILE_BONUS = 1.5
        self.OBSTRUCTION_PENALTY = 0.25
        
        # === TRANSITIONS & DEBT ===
        self.TRANSITION_ATTRACTION_WEIGHT = 0.6
        self.TEMP_DEBT_ACCUMULATION = 0.5
        self.TEMP_DEBT_DECAY = 0.02
        self.TEMP_DEBT_MAX = 15.0
        self.MAX_MAP_DEBT = 10.0
        self.MAX_LOCATION_DEBT = 5.0
        self.DEBT_DECAY_RATE = 0.005
        
        # === TRANSITION BAN ===
        self.transition_bans = {}
        self.BAN_VICINITY_RADIUS = 3
        self.BAN_COVERAGE_LIFT_THRESHOLD = 0.6
        self.BAN_TIMEOUT_STEPS = 300
        
        # Multi-scale memory
        self.visited_maps = {}
        self.map_novelty_debt = {}
        self.location_memory = {}
        self.location_novelty = {}
        self.action_execution_count = {}
        self.MAX_LOCATIONS = 500
        
        self.swap_perceptron = ControlSwapPerceptron()
        self.error_history = deque(maxlen=100)
        self.numeric_error_history = deque(maxlen=100)
        self.visual_error_history = deque(maxlen=100)
        self._entity_norms_cache = {}
        self._cache_valid = False
        self.innate_entities_spawned = False
        
        # === ENTITY SPAWNING & CLUSTERING (matches AI agent) ===
        self.ENTITY_INITIAL_CAPACITY = 20
        self.entity_capacity = self.ENTITY_INITIAL_CAPACITY
        self.ENTITY_CAPACITY_GROWTH = 1.5
        self.ENTITY_CLUSTER_SIMILARITY = 0.85
        self.ENTITY_MIN_ACTIVATIONS = 10
        self.entity_spawn_count = 0
        self.entity_merge_count = 0
        
        # === REPETITION ===
        self.consecutive_action_count = 0
        self.current_repeated_action = None
        self.LEARNING_SLOWDOWN_START = 3
        self.LEARNING_SLOWDOWN_MAX = 10
        self.PENALTY_THRESHOLD = 12
        self.HARD_RESET_THRESHOLD = 18
        
        # === PATTERN ===
        self.PATTERN_CHECK_WINDOW = 50
        self.PATTERN_MIN_REPEATS = 3
        self.PATTERN_MAX_LENGTH = 10
        self.detected_pattern = None
        self.pattern_repeat_count = 0
        
        # === PROBE CACHE ===
        self._cached_probe_action = None
        self._cached_probe_dir = None
        self._probe_cache_position = None

    # =========================================================================
    # HUMAN ACTION TRACKING (NEW)
    # =========================================================================

    def set_human_action(self, action_name):
        """
        Set the human's actual button press for this frame.
        
        Called each frame from the main loop BEFORE learn().
        This ensures brain.last_action reflects the human's real decision,
        so perceptron learning updates are directed to the correct action.
        Also feeds into recent_actions_buffer for Markov matching and
        action_history for pattern detection.
        """
        if action_name is None:
            return
        
        # Expand short codes if needed
        expanded = ACTION_MAP.get(action_name, action_name)
        if expanded:
            expanded = expanded.upper()
        
        # Normalize common variants
        if expanded == "START":
            expanded = "Start"
        elif expanded == "SELECT":
            expanded = "Select"
        
        self.human_action = expanded
        self.last_action = expanded
        self.recent_actions_buffer.append(expanded)
        self.record_action_execution(expanded)
        self.track_consecutive_action(expanded)

    def get_recent_actions_list(self, count=15):
        """Get last N actions as a list. Used for Markov matching in bag/battle."""
        actions = list(self.recent_actions_buffer)
        if len(actions) > count:
            return actions[-count:]
        return actions

    # =========================================================================
    # PARTY DATA MANAGEMENT (NEW — from "pa" field)
    # =========================================================================

    def update_party_data(self, party_data):
        """Update party state from Lua "pa" field. Called every frame."""
        self.prev_party_data = {
            'count': self.party_data.get('count', 0),
            'slots': [s.copy() for s in self.party_data.get('slots', [])],
        }
        self.party_data = {
            'count': party_data.get('count', 0),
            'slots': [s.copy() for s in party_data.get('slots', [])],
        }

    def get_party_hp_ratios(self):
        """Get HP/maxHP ratios for all party members."""
        ratios = []
        for slot in self.party_data.get('slots', []):
            max_hp = slot.get('max_hp', 0)
            if max_hp > 0:
                ratios.append(slot.get('hp', 0) / max_hp)
            else:
                ratios.append(0.0)
        return ratios

    def get_party_status_flags(self):
        """Get status condition flags for all party members."""
        return [slot.get('status', 0) for slot in self.party_data.get('slots', [])]

    def get_party_levels(self):
        """Get levels for all party members."""
        return [slot.get('level', 0) for slot in self.party_data.get('slots', [])]

    def get_lowest_hp_ratio(self):
        """Get the lowest HP ratio across living party members."""
        ratios = self.get_party_hp_ratios()
        living_ratios = [r for r in ratios if r > 0.0]
        if not living_ratios:
            return 0.0
        return min(living_ratios)

    def has_status_condition_in_party(self):
        """Check if any living party member has a status condition."""
        for slot in self.party_data.get('slots', []):
            if slot.get('hp', 0) > 0 and slot.get('status', 0) != 0:
                return True
        return False

    def get_party_snapshot(self):
        """
        Get full party snapshot dict for event timeline generation.
        Matches event_timeline.json party_snapshot/party_before/party_after format.
        """
        return {
            'hp_ratios': self.get_party_hp_ratios(),
            'status_flags': self.get_party_status_flags(),
            'count': self.party_data.get('count', 0),
            'levels': self.get_party_levels(),
        }

    def get_party_context_for_bag(self):
        """
        Get party context dict for taught_bag_transitions.json frames.
        Matches the party_context structure expected by AI agent's
        compute_bag_markov_similarity().
        """
        hp_ratios = self.get_party_hp_ratios()
        status_flags = self.get_party_status_flags()
        lowest = self.get_lowest_hp_ratio()
        has_status = self.has_status_condition_in_party()
        count = self.party_data.get('count', 0)
        
        return {
            'hp_ratios': hp_ratios,
            'status_flags': status_flags,
            'lowest_hp_ratio': lowest,
            'has_status': has_status,
            'party_count': count,
        }

    # =========================================================================
    # MENU DATA MANAGEMENT (NEW — from "mu" field)
    # =========================================================================

    def update_menu_data(self, menu_data):
        """Update menu cursor state from Lua "mu" field. Called every frame."""
        self.prev_menu_data = self.menu_data.copy()
        self.menu_data = menu_data.copy()

    # =========================================================================
    # BAG DATA MANAGEMENT (NEW — from "bg" field)
    # =========================================================================

    def update_bag_data(self, bag_data):
        """Update bag state from Lua "bg" field. Called every frame."""
        self.prev_bag_data = {
            'pocket': self.bag_data.get('pocket', -1),
            'cursor': self.bag_data.get('cursor', -1),
            'active': self.bag_data.get('active', 0),
            'items': [item.copy() for item in self.bag_data.get('items', [])],
        }
        self.bag_data = {
            'pocket': bag_data.get('pocket', -1),
            'cursor': bag_data.get('cursor', -1),
            'active': bag_data.get('active', 0),
            'items': [item.copy() for item in bag_data.get('items', [])],
        }

    def get_item_at_cursor(self):
        """Get the item ID at the current bag cursor position."""
        items = self.bag_data.get('items', [])
        cursor = self.bag_data.get('cursor', -1)
        if 0 <= cursor < len(items):
            return items[cursor].get('id', -1)
        return -1

    # =========================================================================
    # BATTLE DATA MANAGEMENT (EXPANDED)
    # =========================================================================

    def update_battle_data(self, battle_data, in_battle):
        """
        Update battle state from Lua memory reads. Called every frame.
        Buffers battle data when in battle for post-processing.
        """
        self.prev_battle_data = self.battle_data.copy()
        # Preserve list fields as copies
        prev_pss = self.battle_data.get('player_stat_stages')
        prev_ess = self.battle_data.get('enemy_stat_stages')
        if isinstance(prev_pss, list):
            self.prev_battle_data['player_stat_stages'] = list(prev_pss)
        if isinstance(prev_ess, list):
            self.prev_battle_data['enemy_stat_stages'] = list(prev_ess)
        
        self.battle_data = battle_data.copy()
        pss = battle_data.get('player_stat_stages')
        ess = battle_data.get('enemy_stat_stages')
        if isinstance(pss, list):
            self.battle_data['player_stat_stages'] = list(pss)
        if isinstance(ess, list):
            self.battle_data['enemy_stat_stages'] = list(ess)
        
        currently_in_battle = in_battle > 0.5
        
        # Track battle start/end
        if currently_in_battle and not self.in_battle_last_frame:
            self.current_battle_id += 1
        
        # Buffer battle data every frame while in battle
        if currently_in_battle:
            self.battle_data_buffer.append({
                'timestep': self.timestep,
                'battle_id': self.current_battle_id,
                'battle_data': battle_data.copy()
            })
        
        self.in_battle_last_frame = currently_in_battle

    def get_battle_data_for_timestep(self, timestep):
        """Look up battle data for a specific timestep from the buffer."""
        for entry in reversed(self.battle_data_buffer):
            if entry['timestep'] == timestep:
                return entry['battle_data']
            if entry['timestep'] < timestep:
                break
        return DEFAULT_BATTLE_DATA.copy()

    def get_all_battle_data_buffer(self):
        """Return the full battle data buffer for post-processing."""
        return self.battle_data_buffer

    # =========================================================================
    # BAG SESSION RECORDING (NEW)
    # Records human bag interactions for taught_bag_transitions.json
    # =========================================================================

    def start_bag_session(self, context):
        """
        Called when bag opens. Starts recording bag frames.
        
        context: "overworld" (gs changed to 14) or "battle" (bg.a became 1)
        """
        self.bag_session_active = True
        self.bag_session_context = context
        self.bag_session_start_timestep = self.timestep
        self.bag_session_frame_count = 0
        self.bag_session_last_action = None
        self.bag_session_items_used = []
        self.bag_session_pockets_visited = set()
        self.bag_session_frames = []
        
        pocket = self.bag_data.get('pocket', -1)
        if pocket >= 0:
            self.bag_session_pockets_visited.add(pocket)
        
        print(f"  🎒 BAG SESSION START: {context} at step {self.timestep}")

    def record_bag_frame(self, action):
        """
        Record one frame during an active bag session.
        Called every frame while bag is open with the human's current action.
        
        Produces frames matching taught_bag_transitions.json format exactly.
        """
        if not self.bag_session_active:
            return
        if action is None:
            return
        
        # Determine batch_type
        if action != self.bag_session_last_action:
            batch_type = "action_change"
            frame_offset = 0
        else:
            batch_type = "steady"
            # Count consecutive steady frames
            frame_offset = 0
            for f in reversed(self.bag_session_frames):
                if f.get('batch_type') == 'steady':
                    frame_offset += 1
                else:
                    break
        
        # Track pocket visits
        pocket = self.bag_data.get('pocket', -1)
        if pocket >= 0:
            self.bag_session_pockets_visited.add(pocket)
        
        # Detect item use: previous action was A, previous mc was 0 (Use)
        prev_mc = self.prev_menu_data.get('mc', -1)
        if self.bag_session_last_action == "A" and prev_mc == 0:
            # Item was used last frame — record which item
            prev_items = self.prev_bag_data.get('items', [])
            prev_cursor = self.prev_bag_data.get('cursor', -1)
            if 0 <= prev_cursor < len(prev_items):
                item_id = prev_items[prev_cursor].get('id', -1)
                if item_id > 0:
                    self.bag_session_items_used.append(item_id)
                    self.bag_sessions_metadata['items_used'].add(item_id)
        
        # Build frame matching taught_bag_transitions.json format
        frame = {
            'action': action,
            'recent_actions': self.get_recent_actions_list(15),
            'state': {
                'gs': self.game_state_raw,
                'pocket': pocket,
                'cursor': self.bag_data.get('cursor', -1),
                'item_id': self.get_item_at_cursor(),
                'mc': self.menu_data.get('mc', -1),
                'mm': self.menu_data.get('mm', -1),
                'in_battle': self.bag_session_context == "battle",
            },
            'party_context': self.get_party_context_for_bag(),
            'batch_type': batch_type,
            'frame_offset': frame_offset,
        }
        
        self.bag_session_frames.append(frame)
        self.bag_session_frame_count += 1
        self.bag_session_last_action = action

    def end_bag_session(self):
        """
        Called when bag closes. Finalizes the session and accumulates data.
        """
        if not self.bag_session_active:
            return
        
        duration = self.timestep - self.bag_session_start_timestep
        n_frames = len(self.bag_session_frames)
        
        if n_frames > 0:
            # Add all frames to accumulated buffer
            self.bag_sessions_accumulated.extend(self.bag_session_frames)
            
            # Update metadata
            self.bag_sessions_metadata['total_bag_frames'] += n_frames
            self.bag_sessions_metadata['bag_sessions_recorded'] += 1
            self.bag_sessions_metadata['pockets_visited'].update(self.bag_session_pockets_visited)
            
            items_str = f" items_used={self.bag_session_items_used}" if self.bag_session_items_used else ""
            pockets_str = f" pockets={list(self.bag_session_pockets_visited)}"
            print(f"  🎒 BAG SESSION END: {self.bag_session_context} | "
                  f"{n_frames} frames, {duration} timesteps{items_str}{pockets_str}")
        else:
            print(f"  🎒 BAG SESSION END: {self.bag_session_context} | empty (0 frames)")
        
        # Reset session state
        self.bag_session_active = False
        self.bag_session_context = "none"
        self.bag_session_start_timestep = 0
        self.bag_session_frame_count = 0
        self.bag_session_last_action = None
        self.bag_session_items_used = []
        self.bag_session_pockets_visited = set()
        self.bag_session_frames = []

    def detect_bag_session_boundaries(self):
        """
        Called every frame from main loop. Detects bag open/close transitions.
        Returns: "opened", "closed", or None
        """
        gs = self.game_state_raw
        prev_gs = self.prev_game_state_raw
        in_battle = self.battle_data.get('battle_cursor', -1) != -1
        bg_active = self.bag_data.get('active', 0)
        prev_bg_active = self.prev_bag_data.get('active', 0)
        
        if self.bag_session_active:
            # Check for bag close
            if not in_battle and gs != 14:
                # Overworld bag closed (gs left 14)
                return "closed"
            if in_battle and bg_active == 0 and prev_bg_active == 0:
                # Battle bag closed (bg.a dropped to 0)
                return "closed"
            if in_battle and not (self.battle_data.get('battle_cursor', -1) != -1):
                # Battle ended while in bag
                return "closed"
            return None
        else:
            # Check for bag open
            if not in_battle and gs == 14 and prev_gs != 14:
                # Overworld bag opened (gs transitioned to 14)
                return "opened_overworld"
            if in_battle and bg_active == 1 and prev_bg_active != 1:
                # Battle bag opened (bg.a became 1)
                return "opened_battle"
            return None

    def get_accumulated_bag_data(self):
        """
        Get all accumulated bag session data for writing to file.
        Returns dict matching taught_bag_transitions.json format.
        """
        return {
            'bag_frames': self.bag_sessions_accumulated,
            'metadata': {
                'total_bag_frames': self.bag_sessions_metadata['total_bag_frames'],
                'bag_sessions_recorded': self.bag_sessions_metadata['bag_sessions_recorded'],
                'items_used': sorted(list(self.bag_sessions_metadata['items_used'])),
                'pockets_visited': sorted(list(self.bag_sessions_metadata['pockets_visited'])),
            }
        }

    def get_bag_recording_stats(self):
        """Return bag recording stats for logging."""
        return {
            'session_active': self.bag_session_active,
            'session_context': self.bag_session_context,
            'session_frames': self.bag_session_frame_count,
            'total_sessions': self.bag_sessions_metadata['bag_sessions_recorded'],
            'total_frames': self.bag_sessions_metadata['total_bag_frames'],
            'items_used': sorted(list(self.bag_sessions_metadata['items_used'])),
            'pockets_visited': sorted(list(self.bag_sessions_metadata['pockets_visited'])),
        }

    # =========================================================================
    # CORE HELPERS (unchanged)
    # =========================================================================
    def add(self, p): self.perceptrons.append(p); self._cache_valid = False
    def actions(self): return [p for p in self.perceptrons if p.kind == "action"]
    def entities(self): return [p for p in self.perceptrons if p.kind == "entity"]
    def get_location_key(self, x, y, mid, bs=5): return (int(mid), int(x//bs)*bs, int(y//bs)*bs)
    def is_near_map_edge(self, x, y): return x < 10 or x > 245 or y < 10 or y > 245
    def record_action_execution(self, a):
        if a: self.action_execution_count[a] = self.action_execution_count.get(a, 0) + 1
    def get_position_stagnation(self):
        if len(self.last_positions) < 2: return 0
        cp = self.last_positions[-1]
        return sum(1 for p in reversed(list(self.last_positions)[:-1]) if p == cp)
    def get_group_weight(self, g): return sum(a.utility for a in self.actions() if a.group == g)
    def log_state(self, ls, cs): self.prev_learning_states.append(ls); self.prev_context_states.append(cs)
    def update_position(self, x, y): self.last_positions.append((int(x), int(y)))

# ============================================================================
# CELL 3 PART 2: Exploration Memory, Markov, Blend, Transitions, Stagnation
# ============================================================================
# NO FUNCTIONAL CHANGES from original Cell 3.
# All methods preserved exactly. Only reorganized into Part 2 boundary.
# ============================================================================

    # =========================================================================
    # MEMORY MANAGEMENT
    # =========================================================================
    def cleanup_memory(self):
        if len(self.location_memory) > self.MAX_LOCATIONS:
            sl = sorted(self.location_memory.items(), key=lambda x: x[1], reverse=True)
            self.location_memory = dict(sl[:self.MAX_LOCATIONS // 2])
            self.location_novelty = {k: v for k, v in self.location_novelty.items() if k in self.location_memory}
        if len(self.exploration_memory) > self.MAX_MAPS_IN_MEMORY:
            self.save_exploration_memory()
            sm = sorted(self.exploration_memory.items(), key=lambda x: x[1].get('last_visited_timestep', 0), reverse=True)
            self.exploration_memory = dict(sm[:self.MAX_MAPS_IN_MEMORY // 2])
        # Trim battle data buffer if too large
        if len(self.battle_data_buffer) > 50000:
            self.battle_data_buffer = self.battle_data_buffer[-25000:]
        # Trim bag accumulated buffer if too large
        if len(self.bag_sessions_accumulated) > 100000:
            self.bag_sessions_accumulated = self.bag_sessions_accumulated[-50000:]
        self._entity_norms_cache.clear(); self._cache_valid = False; gc.collect()

    def get_memory_stats(self):
        return {'exploration_maps': len(self.exploration_memory), 'location_memory': len(self.location_memory),
                'error_history': len(self.error_history), 'perceptrons': len(self.perceptrons),
                'total_tiles': sum(len(m.get('visited_tiles', set())) for m in self.exploration_memory.values()),
                'battle_buffer_size': len(self.battle_data_buffer),
                'bag_accumulated_frames': len(self.bag_sessions_accumulated)}

    # =========================================================================
    # TAUGHT REFERENCE + BLEND
    # =========================================================================
    def load_taught_reference(self, fp):
        try:
            if not Path(fp).exists(): print(f"  No taught reference at {fp}"); return
            with open(fp, 'r') as f: model = json.load(f)
            if "perceptrons" not in model: return
            for sa in model["perceptrons"].get("actions", []):
                an = sa.get("action")
                if an:
                    self.taught_reference['utilities'][an] = sa.get("utility", 1.0)
                    if sa.get("weights_nonzero"):
                        dim = sa.get("weights_shape", 1376); w = np.zeros(dim)
                        for idx, val in sa["weights_nonzero"]:
                            if idx < dim: w[idx] = val
                        self.taught_reference['weights'][an] = w
            self.taught_reference['loaded'] = True
            print(f"  📖 Taught reference loaded: {list(self.taught_reference['utilities'].keys())}")
        except Exception as e: print(f"  ⚠️ Error loading taught reference: {e}")

    def blend_from_taught(self, tier):
        if not self.taught_reference['loaded'] or tier not in self.BLEND_RATIOS: return
        if self.timestep - self.last_blend_timestep < self.BLEND_COOLDOWN: return
        ai_w, tw = self.BLEND_RATIOS[tier]; bw = (tier == 3)
        for a in self.actions():
            if a.action not in self.taught_reference['utilities']: continue
            tu = self.taught_reference['utilities'][a.action]; a.utility = ai_w * a.utility + tw * tu
            if tu > 1.0: a.utility = max(a.utility, tu * 0.5)
            fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
            a.utility = max(min(a.utility, 2.0), fl)
            if bw and a.action in self.taught_reference['weights'] and a.weights is not None:
                taw = self.taught_reference['weights'][a.action]; md = min(len(a.weights), len(taw))
                a.weights[:md] = ai_w * a.weights[:md] + tw * taw[:md]
        self.last_blend_timestep = self.timestep; self.blend_tier = tier; self.blend_count += 1

    def get_blend_tier(self):
        t3 = self.BLEND_TIER_TRIGGERS[3]
        if self.detected_pattern and self.pattern_repeat_count >= t3['pattern_repeats']: return 3
        if self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * t3['state_stagnation_mult']: return 3
        t2 = self.BLEND_TIER_TRIGGERS[2]
        if self.detected_pattern and self.pattern_repeat_count >= t2['pattern_repeats']: return 2
        if self.get_position_stagnation() >= t2['pos_stagnation']: return 2
        if self.consecutive_action_count >= t2['consecutive']: return 2
        t1 = self.BLEND_TIER_TRIGGERS[1]
        if self.detected_pattern and self.pattern_repeat_count >= t1['pattern_repeats']: return 1
        if self.get_position_stagnation() >= t1['pos_stagnation']: return 1
        if self.consecutive_action_count >= t1['consecutive']: return 1
        return 0

    def try_blend_if_needed(self):
        if not self.taught_reference['loaded']: return False
        tier = self.get_blend_tier()
        if tier == 0: return False
        if tier <= self.blend_tier and (self.timestep - self.last_blend_timestep) < self.BLEND_COOLDOWN: return False
        self.blend_from_taught(tier); return True

    # =========================================================================
    # MARKOV
    # =========================================================================
    def load_taught_transitions(self, fp=None):
        fp = fp or TAUGHT_TRANSITIONS_FILE
        try:
            if Path(fp).exists():
                with open(fp, 'r') as f: data = json.load(f)
                self.taught_transitions = []; self.taught_batches = data.get('batches', [])
                for batch in self.taught_batches:
                    bt = batch.get('batch_type', 'steady'); ta = batch.get('trigger_action')
                    for frame in batch.get('frames', []):
                        self.taught_transitions.append({'state': frame.get('state', {}), 'action': frame.get('action'),
                            'recent_actions': frame.get('recent_actions', []), 'frame_offset': frame.get('frame_offset', 0),
                            'batch_type': bt, 'trigger_action': ta})
                self.taught_metadata = data.get('metadata', {})
                print(f"  📚 Taught transitions: {len(self.taught_batches)} batches, {len(self.taught_transitions)} frames")
            else: self.taught_transitions = []; self.taught_batches = []; self.taught_metadata = {}
        except Exception as e:
            print(f"  Error loading transitions: {e}")
            self.taught_transitions = []; self.taught_batches = []; self.taught_metadata = {}

    def extract_partial_context(self, cs, rp=None):
        rx = rp[0] if rp else int(cs[0]*255); ry = rp[1] if rp else int(cs[1]*255); cm = int(cs[2])
        mb = self.get_position_stagnation() > 3
        nt = False; mem = self.get_current_map_memory(cm)
        for t in mem.get('transitions', []):
            tp = tuple(t['position']) if isinstance(t['position'], list) else t['position']
            if abs(rx - tp[0]) + abs(ry - tp[1]) <= 2: nt = True; break
        return {'in_battle': cs[3] > 0.5, 'in_menu': cs[4] > 0.5, 'movement_blocked': mb,
                'near_transition': nt, 'tile_probed': not self.should_interact_at_tile(rx, ry, cm)}

    def compute_markov_similarity(self, cs, rp=None, taught_frames=None):
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        smc = taught_frames is not None
        if not frames: return 0.0, None, -1
        rx = rp[0] if rp else int(cs[0]*255); ry = rp[1] if rp else int(cs[1]*255)
        cm = int(cs[2]); cd = int(cs[5]); ib = cs[3] > 0.5; im = cs[4] > 0.5
        ca = list(self.action_history); cp = self.extract_partial_context(cs, rp)
        bs, ba, bi = 0.0, None, -1
        for idx, tr in enumerate(frames):
            ts = tr.get('state', {}); ta = tr.get('action'); trc = tr.get('recent_actions', []); bt = tr.get('batch_type', 'steady')
            if not ta or ta == "NONE": continue
            isc = 0.0
            if not smc:
                if ts.get('map_id') != cm: continue
            isc += 0.25
            tx, ty = ts.get('x', 0), ts.get('y', 0); pd = abs(rx-tx) + abs(ry-ty)
            if pd == 0: isc += MARKOV_POS_EXACT_BONUS
            elif pd <= 2: isc += MARKOV_POS_NEAR_BONUS
            elif pd <= MARKOV_POS_MAX_DIST: isc += MARKOV_POS_FAR_BONUS
            else: continue
            if ts.get('direction') == cd: isc += 0.2
            tib = ts.get('in_battle', 0) == 1; tim = ts.get('in_menu', 0) == 1
            if tib == ib: isc += 0.1
            if tim == im: isc += 0.1
            ssc = 0.0
            if trc and ca:
                if len(ca) >= 8 and len(trc) >= 8 and list(ca)[-8:] == trc[-8:]: ssc = MARKOV_SEQ_FULL_WEIGHT
                if ssc < MARKOV_SEQ_MEDIUM_WEIGHT and len(ca) >= 5 and len(trc) >= 5 and list(ca)[-5:] == trc[-5:]: ssc = MARKOV_SEQ_MEDIUM_WEIGHT
                if ssc < MARKOV_SEQ_SHORT_WEIGHT and len(ca) >= 3 and len(trc) >= 3 and list(ca)[-3:] == trc[-3:]: ssc = MARKOV_SEQ_SHORT_WEIGHT
            pm = sum(1 for a, b in [(tib, cp['in_battle']), (tim, cp['in_menu'])] if a == b)
            total = MARKOV_IMMEDIATE_WEIGHT * isc + MARKOV_SEQUENTIAL_WEIGHT * ssc + MARKOV_PARTIAL_WEIGHT * (pm / 2)
            if bt == "action_change": total *= 1.2
            if tr.get('frame_offset', 0) == 0: total *= 1.1
            if total > bs: bs, ba, bi = total, ta, idx
        return bs, ba, bi

    def get_markov_action(self, cs, rp=None, taught_frames=None):
        if not self.markov_enabled: return False, None, 0.0
        frames = taught_frames if taught_frames is not None else self.taught_transitions
        if not frames: return False, None, 0.0
        sc, ac, ix = self.compute_markov_similarity(cs, rp, taught_frames=frames)
        self.last_markov_score = sc
        if sc >= MARKOV_FAMILIARITY_THRESHOLD: self.last_markov_action = ac; return True, ac, sc
        return False, None, sc

    # =========================================================================
    # ACTION EXECUTION CONFIRMATION
    # =========================================================================
    def set_pending_action(self, a): self.pending_action = a; self.pending_action_frames = 0
    def confirm_action_executed(self, cs, pcs):
        if self.pending_action is None: return True
        self.pending_action_frames += 1; ae = False
        if pcs is not None:
            if self.pending_action in ["UP","DOWN","LEFT","RIGHT"]:
                ae = cs[0] != pcs[0] or cs[1] != pcs[1] or cs[5] != pcs[5]
            elif self.pending_action in ["A","B","Start","Select"]:
                ae = abs(cs[4]-pcs[4]) > 0.1 or cs[3] != pcs[3] or cs[2] != pcs[2]
        if ae or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES:
            self.last_confirmed_action = self.pending_action; self.pending_action = None; self.pending_action_frames = 0; return True
        return False
    def should_send_new_action(self):
        return self.pending_action is None or self.pending_action_frames >= self.ACTION_CONFIRM_FRAMES

    # =========================================================================
    # EXPLORATION MEMORY
    # =========================================================================
    def load_exploration_memory(self):
        try:
            if self.EXPLORATION_MEMORY_FILE.exists():
                with open(self.EXPLORATION_MEMORY_FILE, 'r') as f: data = json.load(f)
                self.exploration_memory = {}
                for mk, md in data.items():
                    self.exploration_memory[int(mk.replace('map_', ''))] = self._deserialize_map_memory(md)
                print(f"  Loaded exploration: {len(self.exploration_memory)} maps")
            else: self.exploration_memory = {}
        except Exception as e: print(f"  Error loading exploration: {e}"); self.exploration_memory = {}

    def _deserialize_map_memory(self, d):
        ti = {}
        for tk, td in d.get('tile_interactions', {}).items():
            ti[tk] = {'directions_tried': set(td.get('directions_tried', [])),
                      'direction_attempts': {int(k): v for k, v in td.get('direction_attempts', {}).items()},
                      'direction_successes': {int(k): v for k, v in td.get('direction_successes', {}).items()},
                      'exhausted': td.get('exhausted', False)}
        return {'visited_tiles': set(tuple(t) for t in d.get('visited_tiles', [])),
                'obstructions': set(tuple(t) for t in d.get('obstructions', [])),
                'interactable_objects': d.get('interactable_objects', []),
                'last_visited_timestep': d.get('last_visited_timestep', 0),
                'transitions': d.get('transitions', []), 'temp_debt': d.get('temp_debt', 0.0),
                'tile_interactions': ti}

    def save_exploration_memory(self):
        try:
            data = {f'map_{mid}': self._serialize_map_memory(md) for mid, md in self.exploration_memory.items()}
            with open(self.EXPLORATION_MEMORY_FILE, 'w') as f: json.dump(data, f)
        except Exception as e: print(f"  Error saving exploration: {e}")

    def _serialize_map_memory(self, d):
        sti = {}
        for tk, td in d.get('tile_interactions', {}).items():
            sti[tk] = {'directions_tried': list(td.get('directions_tried', set())),
                       'direction_attempts': {str(k): v for k, v in td.get('direction_attempts', {}).items()},
                       'direction_successes': {str(k): v for k, v in td.get('direction_successes', {}).items()},
                       'exhausted': td.get('exhausted', False)}
        return {'visited_tiles': [list(t) for t in d['visited_tiles']], 'obstructions': [list(t) for t in d['obstructions']],
                'interactable_objects': d['interactable_objects'], 'last_visited_timestep': d['last_visited_timestep'],
                'transitions': d.get('transitions', []), 'temp_debt': d.get('temp_debt', 0.0), 'tile_interactions': sti}

    def get_current_map_memory(self, mid):
        if mid not in self.exploration_memory:
            self.exploration_memory[mid] = {'visited_tiles': set(), 'obstructions': set(), 'interactable_objects': [],
                'last_visited_timestep': self.timestep, 'transitions': [], 'temp_debt': 0.0, 'tile_interactions': {}}
        return self.exploration_memory[mid]

    def record_visited_tile(self, x, y, mid):
        m = self.get_current_map_memory(mid); m['visited_tiles'].add((int(x), int(y))); m['last_visited_timestep'] = self.timestep
    def record_obstruction(self, x, y, mid, d):
        dx, dy = self.DIRECTION_DELTAS_INT.get(d, (0,0)); self.get_current_map_memory(mid)['obstructions'].add((int(x+dx), int(y+dy)))

    def merge_taught_exploration(self, fp):
        if not Path(fp).exists(): print(f"  No taught exploration at {fp}"); return
        try:
            with open(fp, 'r') as f: td = json.load(f)
            ta, ia = 0, 0
            for mk, md in td.items():
                mid = int(mk.replace('map_', '')); tm = self._deserialize_map_memory(md); am = self.get_current_map_memory(mid)
                am['visited_tiles'].update(tm['visited_tiles']); am['obstructions'].update(tm['obstructions'])
                for tt in tm.get('transitions', []):
                    tp = tuple(tt['position']) if isinstance(tt['position'], list) else tt['position']
                    if not any((tuple(e['position']) if isinstance(e['position'], list) else e['position']) == tp and e['direction'] == tt['direction'] for e in am['transitions']):
                        am['transitions'].append(tt); ta += 1
                for ti in tm.get('interactable_objects', []):
                    if ti not in am['interactable_objects']: am['interactable_objects'].append(ti); ia += 1
            print(f"  Merged: {ta} transitions, {ia} interactables")
        except Exception as e: print(f"  Error merging: {e}")

    # =========================================================================
    # TILE INTERACTION
    # =========================================================================
    def get_tile_interaction_key(self, x, y): return f"{int(x)}_{int(y)}"
    def get_tile_interaction_state(self, x, y, mid):
        m = self.get_current_map_memory(mid); tk = self.get_tile_interaction_key(x, y)
        if tk not in m['tile_interactions']:
            m['tile_interactions'][tk] = {'directions_tried': set(), 'direction_attempts': {0:0,1:0,2:0,3:0},
                'direction_successes': {0:0,1:0,2:0,3:0}, 'exhausted': False}
        return m['tile_interactions'][tk]
    def should_interact_at_tile(self, x, y, mid):
        ts = self.get_tile_interaction_state(x, y, mid)
        if ts['exhausted']: return False
        if len(ts['directions_tried']) < 4: return True
        return any(ts['direction_attempts'].get(d,0) > 0 and ts['direction_successes'].get(d,0)/ts['direction_attempts'][d] >= self.MIN_SUCCESS_RATE_THRESHOLD for d in range(4))
    def get_untried_directions(self, x, y, mid):
        return [d for d in range(4) if d not in self.get_tile_interaction_state(x, y, mid)['directions_tried']]
    def get_best_interaction_direction(self, x, y, mid):
        ts = self.get_tile_interaction_state(x, y, mid)
        u = self.get_untried_directions(x, y, mid)
        if u: return u[0]
        bd, br = None, 0.0
        for d in range(4):
            a = ts['direction_attempts'].get(d, 0)
            if a > 0:
                r = ts['direction_successes'].get(d, 0) / a
                if r > br: br, bd = r, d
        return bd
    def get_best_probe_action(self, rx, ry, cm, cd):
        ck = (rx, ry, cm, cd)
        if self._probe_cache_position == ck: return self._cached_probe_action, self._cached_probe_dir
        if not self.should_interact_at_tile(rx, ry, cm): r = (None, None)
        else:
            u = self.get_untried_directions(rx, ry, cm)
            if not u:
                bd = self.get_best_interaction_direction(rx, ry, cm)
                r = ('A', cd) if bd is not None and cd == bd else (self.INT_TO_ACTION[bd], bd) if bd is not None else (None, None)
            elif cd in u: r = ('A', cd)
            else: r = (self.INT_TO_ACTION[u[0]], u[0])
        self._probe_cache_position = ck; self._cached_probe_action, self._cached_probe_dir = r; return r
    def record_tile_interaction_attempt(self, x, y, mid, d, success):
        ts = self.get_tile_interaction_state(x, y, mid); ts['directions_tried'].add(d)
        ts['direction_attempts'][d] = ts['direction_attempts'].get(d, 0) + 1
        if success:
            ts['direction_successes'][d] = ts['direction_successes'].get(d, 0) + 1
            m = self.get_current_map_memory(mid); dn = self.DIRECTION_NAMES.get(d, str(d))
            io = [int(x), int(y), dn]
            if io not in m['interactable_objects']: m['interactable_objects'].append(io); print(f"  🎯 INTERACTABLE: ({x},{y}) {dn}")
        self._check_tile_exhaustion(x, y, mid)
    def _check_tile_exhaustion(self, x, y, mid):
        ts = self.get_tile_interaction_state(x, y, mid)
        if len(ts['directions_tried']) >= 4 and not any(ts['direction_successes'].get(d,0) > 0 for d in range(4)):
            ts['exhausted'] = True
    def start_interaction_verification(self, x, y, mid, d):
        self.pending_interaction_verify = {'x': x, 'y': y, 'map_id': mid, 'direction': d}
        self.interaction_verify_countdown = self.INTERACTION_VERIFY_FRAMES
    def check_interaction_verification(self, cs, pcs):
        if self.pending_interaction_verify is None: return
        self.interaction_verify_countdown -= 1; success = False
        if pcs is not None:
            in_overworld = pcs[3] <= 0.5 and pcs[4] <= 0.5
            if in_overworld:
                success = abs(cs[4]-pcs[4]) > 0.1 or (cs[3] > 0.5 and pcs[3] <= 0.5) or int(cs[2]) != int(pcs[2])
        if success or self.interaction_verify_countdown <= 0:
            i = self.pending_interaction_verify
            self.record_tile_interaction_attempt(i['x'], i['y'], i['map_id'], i['direction'], success)
            self.pending_interaction_verify = None
    def get_tile_interaction_stats(self, mid):
        m = self.get_current_map_memory(mid); ti = m.get('tile_interactions', {})
        return {'probed': len(ti), 'exhausted': sum(1 for t in ti.values() if t.get('exhausted', False)),
                'with_success': sum(1 for t in ti.values() if any(t.get('direction_successes',{}).get(d,0) > 0 for d in range(4)))}
    def get_exploration_coverage(self, mid):
        m = self.get_current_map_memory(mid); v = len(m['visited_tiles']); o = len(m['obstructions'])
        return v / (v + o) if v > 0 and v + o >= 10 else 0.0

    # =========================================================================
    # TRANSITIONS, BANS, DEBT
    # =========================================================================
    def record_transition(self, fp, fm, tm, d, at):
        m = self.get_current_map_memory(fm)
        for t in m['transitions']:
            if t['position'] == list(fp) and t['direction'] == d: t['use_count'] += 1; t['last_used'] = self.timestep; return
        m['transitions'].append({'position': list(fp), 'direction': d, 'action': at, 'destination_map': tm, 'use_count': 1, 'last_used': self.timestep})
        print(f"  🚪 TRANSITION: Map {fm} ({fp}) → Map {tm}")
    def get_transition_attraction(self, cm):
        m = self.get_current_map_memory(cm); ts = m.get('transitions', [])
        if not ts: return 0.0, None
        cd = self.map_novelty_debt.get(cm, 0.0); ctd = self.get_temp_debt(cm); cc = self.get_exploration_coverage(cm)
        ba, bt = 0.0, None
        for t in ts:
            if self.is_transition_banned(cm, t['position'], t['direction']): continue
            dm = t['destination_map']; dd = self.map_novelty_debt.get(dm, 0.0); dtd = self.get_temp_debt(dm); dc = self.get_exploration_coverage(dm)
            a = (cd + ctd*2.0 - dd - dtd*2.0)*0.5 + (cc - dc)*0.5
            if t['use_count'] < 3: a *= 1.5
            if a > ba: ba, bt = a, t
        return ba * self.TRANSITION_ATTRACTION_WEIGHT, bt
    def create_transition_ban(self, mid, tp, db):
        self.transition_bans[mid] = {'banned_tile': tp, 'banned_direction': db, 'vicinity_radius': self.BAN_VICINITY_RADIUS,
            'vicinity_active': False, 'created_at': self.timestep}
    def is_transition_banned(self, mid, pos, d):
        if mid not in self.transition_bans: return False
        b = self.transition_bans[mid]; bt = tuple(b['banned_tile']) if isinstance(b['banned_tile'], list) else b['banned_tile']
        pos = tuple(pos) if isinstance(pos, list) else pos
        if pos == bt and d == b['banned_direction']: return True
        if b['vicinity_active'] and abs(pos[0]-bt[0])+abs(pos[1]-bt[1]) <= b['vicinity_radius'] and d == b['banned_direction']: return True
        return False
    def is_position_banned(self, mid, x, y, d): return self.is_transition_banned(mid, (x,y), d)
    def update_transition_ban(self, mid, cp):
        if mid not in self.transition_bans: return
        b = self.transition_bans[mid]; bt = tuple(b['banned_tile']) if isinstance(b['banned_tile'], list) else b['banned_tile']
        if not b['vicinity_active'] and abs(cp[0]-bt[0])+abs(cp[1]-bt[1]) >= 3: b['vicinity_active'] = True
    def check_ban_lift_conditions(self, mid):
        if mid not in self.transition_bans: return
        b = self.transition_bans[mid]; m = self.get_current_map_memory(mid)
        nb = [t for t in m.get('transitions',[]) if not self.is_transition_banned(mid, t['position'], t['direction'])]
        if nb or self.get_exploration_coverage(mid) >= self.BAN_COVERAGE_LIFT_THRESHOLD or self.timestep - b['created_at'] >= self.BAN_TIMEOUT_STEPS:
            del self.transition_bans[mid]
    def get_temp_debt(self, mid):
        m = self.get_current_map_memory(mid); rd = m.get('temp_debt', 0.0)
        if mid != self.current_map_id:
            return max(0.0, rd - (self.timestep - m.get('last_visited_timestep', 0)) * self.TEMP_DEBT_DECAY)
        return rd
    def accumulate_temp_debt(self, mid):
        m = self.get_current_map_memory(mid); m['temp_debt'] = min(self.TEMP_DEBT_MAX, m.get('temp_debt', 0.0) + self.TEMP_DEBT_ACCUMULATION)
    def decay_all_debts(self):
        for mid in list(self.map_novelty_debt.keys()):
            if mid != self.current_map_id:
                self.map_novelty_debt[mid] *= (1.0 - self.DEBT_DECAY_RATE)
                if self.map_novelty_debt[mid] < 0.1: del self.map_novelty_debt[mid]
    def detect_obstruction(self, pc, cs, rp, prp):
        if pc is None or prp is None or self.last_action not in ['UP','DOWN','LEFT','RIGHT']: return False
        if rp == prp: self.record_obstruction(rp[0], rp[1], int(cs[2]), int(cs[5])); return True
        return False

    # =========================================================================
    # MENU TRAP
    # =========================================================================
    def update_menu_trap_tracking(self, cs, at, rp=None):
        cp = rp if rp else (round(cs[0]*255), round(cs[1]*255))
        if self.menu_trap_position is not None and cp != self.menu_trap_position: self.reset_menu_trap_boost(); return
        if self.get_context_state_hash(cs) == self.last_context_state_hash and at in ["A","B","Start","Select"]:
            self.menu_trap_frames += 1; self.menu_trap_position = cp
            if self.menu_trap_frames > self.MENU_TRAP_THRESHOLD:
                if self.original_b_utility is None:
                    for a in self.actions():
                        if a.action == 'B': self.original_b_utility = a.utility; break
                self.menu_trap_b_boost = min(self.B_BOOST_MAX, self.menu_trap_b_boost + self.B_BOOST_INCREMENT)
        elif cp != self.menu_trap_position: self.reset_menu_trap_boost()
    def reset_menu_trap_boost(self):
        if self.menu_trap_b_boost > 1.0 and self.original_b_utility is not None:
            for a in self.actions():
                if a.action == 'B': a.utility = self.original_b_utility; break
        self.menu_trap_frames = 0; self.menu_trap_b_boost = 1.0; self.menu_trap_position = None; self.original_b_utility = None

    # =========================================================================
    # STAGNATION, PATTERN, MODE
    # =========================================================================
    def get_context_state_hash(self, cs):
        return (round(cs[0],2), round(cs[1],2), int(cs[2]), int(cs[3]), round(cs[4],2), int(cs[5]))
    def check_state_stagnation(self, cs):
        ch = self.get_context_state_hash(cs)
        if ch == self.last_context_state_hash:
            self.state_stagnation_count += 1
            if self.state_stagnation_count == 1 and self.last_action: self.stagnation_initiator_action = self.last_action
        else: self.state_stagnation_count = 0; self.stagnation_initiator_action = None
        self.last_context_state_hash = ch
        return self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD
    def apply_stagnation_initiator_penalty(self):
        if self.stagnation_initiator_action is None: return
        for a in self.actions():
            if a.action == self.stagnation_initiator_action:
                fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                a.utility = max(fl, a.utility * 0.5); break
        self.stagnation_initiator_action = None
    def should_force_random(self):
        f = self.get_position_stagnation() >= 8 or self.consecutive_action_count >= 15 or \
            (self.detected_pattern and self.pattern_repeat_count >= 4) or \
            self.state_stagnation_count >= self.STATE_STAGNATION_THRESHOLD * 2
        if f: self.try_blend_if_needed()
        return f
    def get_forced_random_action_name(self):
        c = ["UP","DOWN","LEFT","RIGHT","A","B"]
        if self.current_repeated_action in c: c.remove(self.current_repeated_action)
        if self.detected_pattern:
            for a in self.detected_pattern:
                if a in c: c.remove(a)
        return random.choice(c or ["UP","DOWN","LEFT","RIGHT"])
    def check_productive_change(self, cs):
        cm = int(cs[2]); cb = cs[3] > 0.5; cp = (cs[0], cs[1]); p, r = False, ""
        if self.last_map_id is not None and cm != self.last_map_id: p, r = True, "map change"
        if self.last_battle_state is not None and cb != self.last_battle_state: p, r = True, "battle change"
        if self.position_at_mode_swap is not None:
            d = np.sqrt((cp[0]-self.position_at_mode_swap[0])**2 + (cp[1]-self.position_at_mode_swap[1])**2)
            if d > 0.03: p, r = True, f"moved {d*255:.1f}"
        cd = int(cs[5])
        if self.direction_change_counts_as_progress and self.last_direction_for_progress is not None and cd != self.last_direction_for_progress:
            self.state_stagnation_count = max(0, self.state_stagnation_count - 5)
        self.last_direction_for_progress = cd; self.last_map_id = cm; self.last_battle_state = cb
        return p, r
    def on_productive_change(self, r):
        self.move_to_interact_threshold = self.DEFAULT_MOVE_TO_INTERACT_THRESHOLD
        self.interact_to_move_threshold = self.DEFAULT_INTERACT_TO_MOVE_THRESHOLD
        self.swap_chain_count = 0; self.state_stagnation_count = 0; self.stagnation_initiator_action = None; self.unproductive_swap_count = 0
        if self.blend_tier > 0: self.blend_tier = 0
    def on_mode_swap(self, fm, tm):
        self.swap_chain_count += 1; self.frames_in_current_mode = 0; self.unproductive_swap_count += 1
        if self.unproductive_swap_count >= self.UNPRODUCTIVE_SWAP_THRESHOLD:
            self._reset_highest_to_third(tm); self.unproductive_swap_count = 0
        if tm == "interact": self.interact_to_move_threshold = min(self.MAX_THRESHOLD, self.interact_to_move_threshold + self.THRESHOLD_INCREMENT)
        else: self.move_to_interact_threshold = min(self.MAX_THRESHOLD, self.move_to_interact_threshold + self.THRESHOLD_INCREMENT)
    def _reset_highest_to_third(self, mode):
        if mode in ["battle","both"]: return
        g = "move" if mode == "move" else "interact"; ga = sorted([a for a in self.actions() if a.group == g], key=lambda a: a.utility, reverse=True)
        if len(ga) >= 3:
            fl = self.INTERACT_UTILITY_FLOOR if g == "interact" else self.MOVE_UTILITY_FLOOR
            ga[0].utility = max(ga[2].utility * 0.9, fl)
    def should_use_both_mode(self):
        return self.state_stagnation_count > self.BOTH_MODE_STAGNATION_THRESHOLD or self.unproductive_swap_count > self.BOTH_MODE_SWAP_THRESHOLD
    def determine_control_mode(self, cs, raw_position=None):
        if cs[3] > 0.5: return "battle"
        self.frames_in_current_mode += 1; ps = self.get_position_stagnation()
        p, r = self.check_productive_change(cs)
        if p: self.on_productive_change(r)
        if self.should_use_both_mode(): return "both"
        if self.check_state_stagnation(cs):
            self.apply_stagnation_initiator_penalty()
            nm = "interact" if self.control_mode == "move" else "move"
            self.control_mode = nm; self.position_at_mode_swap = (cs[0], cs[1])
            self.on_mode_swap(self.control_mode, nm); self.state_stagnation_count = 0; return self.control_mode
        rx = raw_position[0] if raw_position else int(cs[0]*255); ry = raw_position[1] if raw_position else int(cs[1]*255); cm = int(cs[2])
        tp = self.should_interact_at_tile(rx, ry, cm); ud = self.get_untried_directions(rx, ry, cm)
        if tp and ud and self.control_mode == "move" and self.frames_in_current_mode >= 3:
            self.control_mode = "interact"; self.position_at_mode_swap = (cs[0], cs[1]); self.frames_in_current_mode = 0; return self.control_mode
        if self.control_mode == "move" and ps >= self.move_to_interact_threshold:
            self.control_mode = "interact"; self.position_at_mode_swap = (cs[0], cs[1]); self.on_mode_swap("move", "interact")
        elif self.control_mode == "interact":
            if (not tp or not ud) and self.frames_in_current_mode >= 5:
                self.control_mode = "move"; self.position_at_mode_swap = (cs[0], cs[1]); self.frames_in_current_mode = 0
            elif self.frames_in_current_mode >= self.interact_to_move_threshold:
                self.control_mode = "move"; self.position_at_mode_swap = (cs[0], cs[1]); self.on_mode_swap("interact", "move")
        return self.control_mode

    # =========================================================================
    # REPETITION & PATTERN
    # =========================================================================
    def track_consecutive_action(self, an):
        if an == self.current_repeated_action: self.consecutive_action_count += 1
        else: self.current_repeated_action = an; self.consecutive_action_count = 1
    def get_learning_multiplier(self, an):
        if an != self.current_repeated_action or self.consecutive_action_count < self.LEARNING_SLOWDOWN_START: return 1.0
        return max(0.05, 1.0 - 0.95 * min(1.0, (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) / (self.LEARNING_SLOWDOWN_MAX - self.LEARNING_SLOWDOWN_START)))
    def detect_pattern(self):
        if len(self.action_history) < 6: return None, 0
        recent = list(self.action_history)[-self.PATTERN_CHECK_WINDOW:]
        for pl in range(1, self.PATTERN_MAX_LENGTH + 1):
            if len(recent) < pl * self.PATTERN_MIN_REPEATS: continue
            cand = tuple(recent[-pl:]); rc, ix = 0, len(recent) - pl
            while ix >= 0 and tuple(recent[ix:ix+pl]) == cand: rc += 1; ix -= pl
            if rc >= self.PATTERN_MIN_REPEATS: return cand, rc
        return None, 0
    def apply_pattern_penalty(self):
        pat, rc = self.detect_pattern()
        if pat is None: self.detected_pattern = None; self.pattern_repeat_count = 0; return
        self.detected_pattern, self.pattern_repeat_count = pat, rc
        pf = max(0.3, 1.0 - rc * 0.15)
        for an in set(pat):
            for a in self.actions():
                if a.action == an:
                    fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                    a.utility = max(fl, a.utility * pf); break
    def apply_repetition_penalty(self):
        if self.current_repeated_action is None: return
        for a in self.actions():
            if a.action == self.current_repeated_action:
                fl = self.INTERACT_UTILITY_FLOOR if a.group == "interact" else self.MOVE_UTILITY_FLOOR
                if self.consecutive_action_count >= self.HARD_RESET_THRESHOLD:
                    a.utility = fl; self.consecutive_action_count = 0
                elif self.consecutive_action_count >= self.PENALTY_THRESHOLD:
                    a.utility = max(a.utility * 0.5, fl)
                break

    # =========================================================================
    # EXPLORATION TRACKING
    # =========================================================================
    def update_exploration_tracking(self, cs, pcs, rp=None, prp=None):
        cm = int(cs[2]); rx = rp[0] if rp else int(cs[0]*255); ry = rp[1] if rp else int(cs[1]*255); cp = (rx, ry)
        if self.current_map_id is not None and cm != self.current_map_id:
            pm = self.current_map_id
            if pcs is not None and prp is not None:
                self.record_transition(prp, pm, cm, int(pcs[5]), 'interact' if self.last_action == 'A' else 'walk')
            if prp is not None:
                ed = int(cs[5]) if pcs is not None else 0
                self.create_transition_ban(cm, cp, (ed + 2) % 4)
            self.on_map_change(cm)
        self.current_map_id = cm; self.record_visited_tile(rx, ry, cm); self.accumulate_temp_debt(cm)
        self.update_transition_ban(cm, cp); self.check_ban_lift_conditions(cm)
        if pcs is not None and prp is not None: self.detect_obstruction(pcs, cs, rp, prp)
        self.check_interaction_verification(cs, pcs); self.last_direction = int(cs[5])
        if self.timestep % 300 == 0: self.decay_all_debts()
    def on_map_change(self, nm):
        self.save_exploration_memory(); self.control_mode = "move"; self.frames_in_current_mode = 0
        m = self.get_current_map_memory(nm)
        print(f"  🗺️ MAP CHANGE → {nm}: {len(m['visited_tiles'])} visited, {len(m['obstructions'])} obs")

# ============================================================================
# CELL 3 PART 3: Entity Spawning/Clustering, Learning, Save/Load
# ============================================================================
# CHANGES FROM PREVIOUS:
# 1. NEW: spawn_entity_from_novelty() — spawns entity perceptrons from novel
#    states, matching AI agent's implementation exactly
# 2. NEW: check_entity_capacity() — triggers clustering when entity count
#    exceeds capacity, expands capacity if clustering isn't sufficient
# 3. NEW: cluster_entities() — cosine similarity clustering on activation
#    patterns, merges similar entities by averaging weights
# 4. NEW: get_spawn_threshold_adaptive() — percentile-based spawn threshold
# 5. UPDATED: learn() — now properly uses brain.last_action (set from human
#    input by set_human_action()) for directed perceptron updates. The action
#    perceptron matching the human's choice gets reinforced in the states
#    where the human chose it. Entity spawning triggers during teaching.
# 6. UPDATED: save_model_checkpoint() — includes full entity data with
#    cluster_activations info, entity spawn/merge counts, bag recording stats
# 7. UPDATED: load_taught_model() — restores entities from checkpoint
# ============================================================================

    # =========================================================================
    # ENTITY SPAWNING & CLUSTERING (matches AI agent)
    # =========================================================================

    def spawn_entity_from_novelty(self, learning_state, context_state, raw_position=None):
        """
        Spawn a new entity perceptron from the current novel state.
        The entity's weights are initialized from the state that surprised the system,
        so it learns to detect similar situations in the future.
        
        This is the core of empirical activation discovery:
        - The weight vector IS the learned activation function
        - dot(weights, state) fires high when the current state resembles
          the novel state this entity was born from
        - Over time, update() refines the weights based on prediction errors
        - Clustering merges entities with similar activation patterns
        """
        entity = Perceptron("entity", entity_type=f"spawned_{self.entity_spawn_count}")
        entity.ensure_weights(len(learning_state))

        state_norm = np.linalg.norm(learning_state)
        if state_norm > 0:
            entity.weights = (learning_state / state_norm) * 0.1
        else:
            entity.weights = np.random.randn(len(learning_state)) * 0.001

        entity.utility = 1.0
        self.add(entity)
        self.entity_spawn_count += 1

        self.check_entity_capacity()

    def check_entity_capacity(self):
        """
        If entity count exceeds capacity, run clustering.
        If clustering didn't reduce count enough, expand capacity.
        """
        n_entities = len(self.entities())

        if n_entities < self.entity_capacity:
            return

        before_count = n_entities
        self.cluster_entities()
        after_count = len(self.entities())

        if after_count >= before_count * 0.9:
            old_cap = self.entity_capacity
            self.entity_capacity = int(self.entity_capacity * self.ENTITY_CAPACITY_GROWTH)
            print(f"  🧩 Entity capacity expanded: {old_cap} → {self.entity_capacity} "
                  f"(clustering only reduced {before_count} → {after_count})")

    def cluster_entities(self):
        """
        Cluster similar entity perceptrons using cosine similarity on activation patterns.
        Merge similar entities by averaging their weights.
        
        This is how empirically discovered activation functions get refined:
        - Multiple entities that fire on similar states get merged
        - The merged entity's weights = average = a more robust detector
        - Innate entities (sense_menu, sense_battle, etc.) are never merged
        """
        entities = self.entities()

        innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"}
        spawned = [e for e in entities if e.entity_type not in innate_types]
        innate = [e for e in entities if e.entity_type in innate_types]

        if len(spawned) < 2:
            return

        # Split into clusterable (enough activations) and too_young
        clusterable = []
        too_young = []

        for e in spawned:
            if len(e.cluster_activations) >= self.ENTITY_MIN_ACTIVATIONS:
                clusterable.append(e)
            else:
                too_young.append(e)

        if len(clusterable) < 2:
            return

        # Build activation vectors for comparison
        max_len = max(len(e.cluster_activations) for e in clusterable)
        activation_vecs = []
        for e in clusterable:
            vec = list(e.cluster_activations)
            while len(vec) < max_len:
                vec.append(0.0)
            activation_vecs.append(np.array(vec))

        # Find merge groups via cosine similarity
        merged_indices = set()
        merge_groups = []

        for i in range(len(clusterable)):
            if i in merged_indices:
                continue
            group = [i]
            vec_i = activation_vecs[i]
            norm_i = np.linalg.norm(vec_i)
            if norm_i < 1e-10:
                continue

            for j in range(i + 1, len(clusterable)):
                if j in merged_indices:
                    continue
                vec_j = activation_vecs[j]
                norm_j = np.linalg.norm(vec_j)
                if norm_j < 1e-10:
                    continue

                cosine_sim = np.dot(vec_i, vec_j) / (norm_i * norm_j)

                if cosine_sim >= self.ENTITY_CLUSTER_SIMILARITY:
                    group.append(j)
                    merged_indices.add(j)

            if len(group) > 1:
                merged_indices.add(i)
                merge_groups.append(group)

        if not merge_groups:
            return

        # Create merged entities
        new_entities = []
        merged_set = set()

        for group in merge_groups:
            group_entities = [clusterable[idx] for idx in group]

            min_dim = min(len(e.weights) for e in group_entities if e.weights is not None)
            if min_dim == 0:
                continue

            avg_weights = np.zeros(min_dim)
            for e in group_entities:
                avg_weights += e.weights[:min_dim]
            avg_weights /= len(group_entities)

            merged = Perceptron("entity", entity_type=f"merged_{self.entity_merge_count}")
            merged.weights = avg_weights
            merged.utility = max(e.utility for e in group_entities)
            merged.familiarity = np.mean([e.familiarity for e in group_entities])
            merged.learning_rate = np.mean([e.learning_rate for e in group_entities])

            new_entities.append(merged)
            self.entity_merge_count += 1

            for idx in group:
                merged_set.add(id(clusterable[idx]))

        # Rebuild perceptron list
        kept_spawned = [e for e in clusterable if id(e) not in merged_set]

        self.perceptrons = (
            [p for p in self.perceptrons if p.kind == "action"] +
            innate +
            kept_spawned +
            too_young +
            new_entities
        )
        self._cache_valid = False

        total_merged = sum(len(g) for g in merge_groups)
        print(f"  🧩 CLUSTERED: {total_merged} entities → {len(new_entities)} merged "
              f"| Total entities now: {len(self.entities())}")

    # =========================================================================
    # ENTITY & LEARNING
    # =========================================================================

    def spawn_innate_entities(self, ls):
        if self.innate_entities_spawned: return
        for et, ix in [("sense_menu",[5,6]),("sense_battle",[3,4]),("sense_movement",[0,1]),("sense_map_transition",[2])]:
            e = Perceptron("entity", entity_type=et); e.ensure_weights(len(ls)); e.weights = np.zeros(len(ls))
            for i in ix:
                if i < len(e.weights): e.weights[i] = 0.5 if len(ix) > 1 else 1.0
            self.add(e)
        self.innate_entities_spawned = True

    def enforce_utility_floors(self):
        for a in self.actions():
            a.utility = max(a.utility, self.MOVE_UTILITY_FLOOR if a.group == "move" else self.INTERACT_UTILITY_FLOOR)

    def get_spawn_threshold_adaptive(self, error_type='combined', percentile=50):
        """Adaptive spawn threshold based on error history percentile."""
        history = {'numeric': self.numeric_error_history, 'visual': self.visual_error_history}.get(
            error_type, self.error_history)
        return max(0.001, np.percentile(history, percentile)) if len(history) >= 100 else 0.0005

    def stagnation_level(self, w=10):
        if len(self.prev_learning_states) < w: return 0.0
        r = list(self.prev_learning_states)[-w:]
        return 1.0 - np.tanh(np.mean([np.linalg.norm(r[i][:min(len(r[i]),len(r[i-1]))] - r[i-1][:min(len(r[i]),len(r[i-1]))]) for i in range(1, len(r))]) * 2.0)

    def predict_future_error(self, st, ac, cs, rp=None):
        en = np.mean([e.predict(st) * e.utility for e in self.entities()]) if self.entities() else 0.5
        comb = en * 0.7 + ac.utility * 0.3; cm = int(cs[2])
        loc = self.get_location_key(*(rp if rp else (cs[0]*255, cs[1]*255)), cm)
        td = min(self.map_novelty_debt.get(cm, 0.0), self.MAX_MAP_DEBT) + self.get_temp_debt(cm) + min(self.location_novelty.get(loc, 0.0), self.MAX_LOCATION_DEBT) * 0.5
        comb *= 1.0 / (1.0 + td * 5.0)
        if ac.action == self.current_repeated_action and self.consecutive_action_count > self.LEARNING_SLOWDOWN_START:
            comb *= 1.0 / (1.0 + (self.consecutive_action_count - self.LEARNING_SLOWDOWN_START) * 0.15)
        if self.detected_pattern and ac.action in self.detected_pattern:
            comb *= 1.0 / (1.0 + self.pattern_repeat_count * 0.2)
        return comb + np.random.randn() * 0.05

    def compute_multi_modal_error(self, s, ns):
        ml = min(len(s), len(ns)); d = [abs(ns[i]-s[i]) for i in range(min(8, ml))]
        w = [0.5,0.5,10.0,5.0,3.0,2.0,1.5,0.3]
        we = sum(di*wi for di, wi in zip(d, w[:len(d)])) + (np.linalg.norm(ns[8:ml]-s[8:ml])*2.0 if ml > 8 else 0.0)
        return we, sum(d), (np.linalg.norm(ns[8:ml]-s[8:ml]) if ml > 8 else 0.0)

    def learn(self, ls, nls, cs, ncs, dead=False, raw_position=None, next_raw_position=None):
        """
        Core learning method. Called every frame during human play.
        
        KEY DIFFERENCE from previous version:
        brain.last_action is now set from the human's actual button press
        (via set_human_action() called in the main loop before learn()).
        This means:
        - The action perceptron matching the human's choice gets the directed
          learning update (via learning_mult applied to that specific perceptron)
        - Entity spawning works properly because last_action is not None
        - Pattern/repetition detection works on real human input sequences
        - The taught model checkpoint encodes real human decision patterns
        """
        if ls.shape != nls.shape:
            md = max(len(ls), len(nls)); ls = np.pad(ls, (0, max(0, md-len(ls)))); nls = np.pad(nls, (0, max(0, md-len(nls))))
        if not self.innate_entities_spawned: self.spawn_innate_entities(ls)
        pc = self.prev_context_states[-1] if self.prev_context_states else None
        pr = getattr(self, '_last_raw_position', None)
        self.update_exploration_tracking(cs, pc, raw_position, pr); self._last_raw_position = raw_position
        we, ne, ve = self.compute_multi_modal_error(ls, nls)
        self.error_history.append(we); self.numeric_error_history.append(ne); self.visual_error_history.append(ve)

        # === ENTITY SPAWNING: spawn from novel states during human play ===
        # This creates entity perceptrons whose weight vectors encode the
        # sensory patterns the human encountered. These become the empirically
        # discovered activation functions in the taught model.
        spawn_threshold = self.get_spawn_threshold_adaptive('combined', percentile=75)
        if we > spawn_threshold and len(self.error_history) >= 100:
            menu_active = cs[4] > 0.5
            battle_active = cs[3] > 0.5
            if not menu_active and not battle_active:
                self.spawn_entity_from_novelty(ls, cs, raw_position)

        cm = int(cs[2]); loc = self.get_location_key(*(raw_position if raw_position else (cs[0]*255, cs[1]*255)), cm)
        self.visited_maps[cm] = self.visited_maps.get(cm, 0) + 1
        if len(self.location_memory) < self.MAX_LOCATIONS: self.location_memory[loc] = self.location_memory.get(loc, 0) + 1
        if self.visited_maps[cm] > 10: self.map_novelty_debt[cm] = min(self.MAX_MAP_DEBT, self.map_novelty_debt.get(cm, 0.0) + 0.05*(self.visited_maps[cm]-10))
        if self.location_memory.get(loc, 0) > 15: self.location_novelty[loc] = min(self.MAX_LOCATION_DEBT, self.location_novelty.get(loc, 0.0) + 0.1*(self.location_memory.get(loc,0)-15))
        if self.visited_maps[cm] > 30: we *= 0.5
        if self.location_memory.get(loc, 0) > 25: we *= 0.7

        stag = self.stagnation_level()
        
        # === DIRECTED LEARNING from human action ===
        # brain.last_action is set from the human's actual button press.
        # The learning multiplier is applied specifically to the action perceptron
        # that matches what the human chose, so that perceptron's weights get
        # reinforced in the current state. This is how action perceptrons learn
        # "in this state, the human pressed UP" — the UP perceptron's weights
        # shift toward the current learning_state.
        lm = self.get_learning_multiplier(self.last_action) if self.last_action else 1.0
        if self.detected_pattern and self.last_action and self.last_action in self.detected_pattern: lm *= 0.5
        
        for p in self.perceptrons:
            m = lm if (p.kind == "action" and p.action == self.last_action) else 1.0
            if p.kind == "action" and self.detected_pattern and p.action in self.detected_pattern: m *= 0.5
            p.update(ls, we * m, stagnation=stag)
        
        for a in self.actions():
            if a.action in ['Start','Select'] and a.weights is not None: a.weights *= 0.999
        self.apply_repetition_penalty(); self.apply_pattern_penalty(); self.enforce_utility_floors()
        
        # Movement boost: if position changed and last action was a movement
        if pc is not None and np.linalg.norm(cs[:2] - pc[:2]) > 0.001 and self.last_action and self.consecutive_action_count < self.PENALTY_THRESHOLD:
            for a in self.actions():
                if a.action == self.last_action:
                    a.utility = min(a.utility * (1.15 if raw_position and self.is_near_map_edge(*raw_position) else 1.08), 2.0); break
        
        if self.timestep % 1000 == 0: self.cleanup_memory()
        if self.timestep % self.SAVE_INTERVAL == 0: self.save_exploration_memory()
        self.action_history.append(self.last_action)

    # =========================================================================
    # SAVE/LOAD
    # =========================================================================
    def save_model_checkpoint(self, filepath=None):
        """
        Save model checkpoint with full entity data.
        
        The checkpoint includes:
        - Action perceptrons: utilities, weights (sparse), learning rates
        - Entity perceptrons: types, utilities, weights (sparse), familiarity
        - Debt tracking: map debt, location debt, visit counts
        - Control state: mode, Markov stats, blend stats
        - Entity stats: spawn count, merge count, capacity
        - Battle stats: battle count, buffer size
        - Bag stats: sessions recorded, frames accumulated
        
        When the AI agent loads this via initialize_from_taught_model(),
        it gets meaningful action weights AND entity activation patterns
        learned from the human's actual gameplay.
        """
        if filepath is None: filepath = BASE_PATH / "taught_model_checkpoint.json"
        model = {
            "timestep": self.timestep,
            "perceptrons": {"actions": [], "entities": []},
            "debt_tracking": {
                "map_novelty_debt": {str(k): v for k, v in self.map_novelty_debt.items()},
                "location_novelty": {str(k): v for k, v in self.location_novelty.items()},
                "visited_maps": {str(k): v for k, v in self.visited_maps.items()}
            },
            "control_mode": self.control_mode,
            "markov_stats": {
                "markov_action_count": self.markov_action_count,
                "curiosity_action_count": self.curiosity_action_count
            },
            "blend_stats": {
                "blend_count": self.blend_count,
                "last_blend_tier": self.blend_tier
            },
            "battle_stats": {
                "battles_recorded": self.current_battle_id,
                "battle_buffer_size": len(self.battle_data_buffer)
            },
            "entity_stats": {
                "spawn_count": self.entity_spawn_count,
                "merge_count": self.entity_merge_count,
                "capacity": self.entity_capacity,
                "total_entities": len(self.entities()),
                "innate_spawned": self.innate_entities_spawned,
            },
            "bag_stats": {
                "sessions_recorded": self.bag_sessions_metadata['bag_sessions_recorded'],
                "total_frames": self.bag_sessions_metadata['total_bag_frames'],
                "items_used": sorted(list(self.bag_sessions_metadata['items_used'])),
                "pockets_visited": sorted(list(self.bag_sessions_metadata['pockets_visited'])),
            },
        }
        
        for a in self.actions():
            model["perceptrons"]["actions"].append({
                "action": a.action, "group": a.group, "utility": float(a.utility),
                "weights_shape": len(a.weights) if a.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(a.weights) if abs(v) > 1e-10] if a.weights is not None else [],
                "learning_rate": float(a.learning_rate), "familiarity": float(a.familiarity)
            })
        
        for e in self.entities():
            entity_data = {
                "entity_type": e.entity_type, "utility": float(e.utility),
                "weights_shape": len(e.weights) if e.weights is not None else 0,
                "weights_nonzero": [[i, float(v)] for i, v in enumerate(e.weights) if abs(v) > 1e-10] if e.weights is not None else [],
                "familiarity": float(e.familiarity),
                "learning_rate": float(e.learning_rate),
                "activation_history_len": len(e.activation_history),
                "cluster_activations_len": len(e.cluster_activations),
            }
            model["perceptrons"]["entities"].append(entity_data)
        
        try:
            with open(filepath, 'w') as f: json.dump(model, f, indent=2)
            n_actions = len(model["perceptrons"]["actions"])
            n_entities = len(model["perceptrons"]["entities"])
            print(f"💾 Model saved: step {self.timestep} | {n_actions} actions, {n_entities} entities → {filepath}")
        except Exception as e: print(f"❌ Save error: {e}")

    def load_taught_model(self, fp):
        """
        Load a previously saved teaching model checkpoint.
        Restores actions, entities, debt tracking, and all stats.
        """
        if not Path(fp).exists(): return 0
        try:
            with open(fp, 'r') as f: model = json.load(f)
            if "perceptrons" not in model: return 0
            
            # Restore action perceptrons
            for sa in model["perceptrons"]["actions"]:
                for a in self.actions():
                    if a.action == sa["action"]:
                        a.utility = sa["utility"]
                        a.learning_rate = sa.get("learning_rate", 0.01)
                        a.familiarity = sa.get("familiarity", 0.0)
                        if sa.get("weights_nonzero"):
                            dim = sa.get("weights_shape", 1376); a.weights = np.zeros(dim)
                            for idx, val in sa["weights_nonzero"]:
                                if idx < dim: a.weights[idx] = val
                        break
            
            # Restore entity perceptrons
            saved_entities = model["perceptrons"].get("entities", [])
            if saved_entities:
                # Remove any existing spawned entities (keep innate if already spawned)
                innate_types = {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"}
                
                for se in saved_entities:
                    et = se.get("entity_type", "unknown")
                    
                    # Check if this is an innate entity that already exists
                    if et in innate_types:
                        existing = None
                        for e in self.entities():
                            if e.entity_type == et:
                                existing = e
                                break
                        if existing:
                            # Update existing innate entity
                            existing.utility = se.get("utility", 1.0)
                            existing.familiarity = se.get("familiarity", 0.0)
                            existing.learning_rate = se.get("learning_rate", 0.01)
                            if se.get("weights_nonzero"):
                                dim = se.get("weights_shape", 1376)
                                existing.ensure_weights(dim)
                                existing.weights = np.zeros(dim)
                                for idx, val in se["weights_nonzero"]:
                                    if idx < dim: existing.weights[idx] = val
                            continue
                    
                    # Create new entity (spawned or merged types)
                    e = Perceptron("entity", entity_type=et)
                    e.utility = se.get("utility", 1.0)
                    e.familiarity = se.get("familiarity", 0.0)
                    e.learning_rate = se.get("learning_rate", 0.01)
                    if se.get("weights_nonzero"):
                        dim = se.get("weights_shape", 1376)
                        e.weights = np.zeros(dim)
                        for idx, val in se["weights_nonzero"]:
                            if idx < dim: e.weights[idx] = val
                    self.add(e)
                
                self.innate_entities_spawned = True
                print(f"  🧩 Restored {len(saved_entities)} entities from checkpoint")
            
            # Restore debt tracking
            if "debt_tracking" in model:
                d = model["debt_tracking"]
                self.map_novelty_debt = {int(k): v for k, v in d.get("map_novelty_debt", {}).items()}
                self.visited_maps = {int(k): v for k, v in d.get("visited_maps", {}).items()}
                for k, v in d.get("location_novelty", {}).items():
                    try: self.location_novelty[eval(k)] = v
                    except: pass
            
            # Restore stats
            ms = model.get("markov_stats", {})
            self.markov_action_count = ms.get("markov_action_count", 0)
            self.curiosity_action_count = ms.get("curiosity_action_count", 0)
            
            bs = model.get("blend_stats", {})
            self.blend_count = bs.get("blend_count", 0)
            self.blend_tier = bs.get("last_blend_tier", 0)
            
            bts = model.get("battle_stats", {})
            self.current_battle_id = bts.get("battles_recorded", 0)
            
            # Restore entity stats
            es = model.get("entity_stats", {})
            self.entity_spawn_count = es.get("spawn_count", 0)
            self.entity_merge_count = es.get("merge_count", 0)
            self.entity_capacity = es.get("capacity", self.ENTITY_INITIAL_CAPACITY)
            if es.get("innate_spawned", False):
                self.innate_entities_spawned = True
            
            # Restore bag stats
            bag_s = model.get("bag_stats", {})
            self.bag_sessions_metadata['bag_sessions_recorded'] = bag_s.get("sessions_recorded", 0)
            self.bag_sessions_metadata['total_bag_frames'] = bag_s.get("total_frames", 0)
            self.bag_sessions_metadata['items_used'] = set(bag_s.get("items_used", []))
            self.bag_sessions_metadata['pockets_visited'] = set(bag_s.get("pockets_visited", []))
            
            self.timestep = model.get("timestep", 0)
            
            n_actions = len(model["perceptrons"]["actions"])
            n_entities = len(model["perceptrons"].get("entities", []))
            print(f"  📖 Model loaded: step {self.timestep} | {n_actions} actions, {n_entities} entities")
            print(f"     Entity stats: {self.entity_spawn_count} spawned, {self.entity_merge_count} merged, capacity {self.entity_capacity}")
            print(f"     Utilities: {[f'{a.action}:{a.utility:.3f}' for a in self.actions()]}")
            
            return self.timestep
        except Exception as e:
            print(f"  ⚠️ Load error: {e}")
            return 0

    def save_model(self, fp=None): self.save_model_checkpoint(fp)
    def load_model(self, fp=None): return self.load_taught_model(fp or (BASE_PATH / "taught_model_checkpoint.json"))

In [ ]:
# ============================================================================
# CELL 6A: Post-Processors (5 total)
# ============================================================================
# 1. run_per_map_analysis() — UNCHANGED
# 2. run_nav_target_extraction() — UNCHANGED
# 3. run_battle_extraction() — UPDATED with expanded battle data fields
# 4. run_bag_extraction() — NEW, writes taught_bag_transitions.json
# 5. run_event_timeline_generation() — NEW, writes event_timeline.json
# 6. run_all_post_processing() — calls all 5
# ============================================================================

from collections import defaultdict
from datetime import datetime

# =========================================================================
# POST-PROCESSOR 1: per_map_analysis (UNCHANGED)
# =========================================================================
def run_per_map_analysis():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — skipping per_map_analysis")
        return
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    batches = data.get('batches', [])
    if not batches:
        print("  ⚠️ No batches — skipping per_map_analysis")
        return
    all_frames = []
    for batch in batches:
        for frame in batch.get('frames', []):
            all_frames.append(frame)
    print(f"  Analyzing {len(all_frames)} frames...")
    frames_by_map = defaultdict(list)
    for i, frame in enumerate(all_frames):
        mid = frame.get('state', {}).get('map_id')
        if mid is not None:
            frames_by_map[mid].append((i, frame))
    per_map = {}
    for map_id, indexed_frames in frames_by_map.items():
        mk = str(map_id)
        ta = defaultdict(lambda: defaultdict(int))
        td = defaultdict(lambda: defaultdict(int))
        tt = defaultdict(int)
        for i, fr in indexed_frames:
            s = fr.get('state', {}); x, y = s.get('x', 0), s.get('y', 0)
            tk = f"{x}_{y}"; act = fr.get('action', 'NONE'); dr = s.get('direction', 0)
            ta[tk][act] += 1; td[tk][str(dr)] += 1; tt[tk] += 1
        ap = {}
        for tk in ta:
            total = tt[tk]
            if total == 0: continue
            probs = {a: round(c/total, 3) for a, c in ta[tk].items()}
            probs['total_frames'] = total; probs['direction_facing'] = dict(td[tk]); ap[tk] = probs
        mg = defaultdict(set)
        for idx in range(len(indexed_frames) - 1):
            _, f1 = indexed_frames[idx]; _, f2 = indexed_frames[idx+1]
            s1, s2 = f1.get('state', {}), f2.get('state', {})
            if s1.get('map_id') != s2.get('map_id'): continue
            x1, y1 = s1.get('x',0), s1.get('y',0); x2, y2 = s2.get('x',0), s2.get('y',0)
            if (x1,y1) != (x2,y2):
                t1, t2 = f"{x1}_{y1}", f"{x2}_{y2}"; mg[t1].add(t2); mg[t2].add(t1)
        mg_s = {k: sorted(list(v)) for k, v in mg.items()}
        dp = []
        for idx in range(1, len(indexed_frames)):
            _, fp = indexed_frames[idx-1]; gi, fc = indexed_frames[idx]
            ap2, ac = fp.get('action','NONE'), fc.get('action','NONE')
            if ap2 != ac and ac != 'NONE' and ap2 != 'NONE':
                s = fc.get('state', {})
                dp.append({'position': [s.get('x',0), s.get('y',0)], 'from_action': ap2, 'to_action': ac,
                    'frame': gi, 'facing': s.get('direction',0),
                    'context': {'in_battle': s.get('in_battle',0), 'in_menu': s.get('in_menu',0)}})
        dd = defaultdict(lambda: {'visits': 0, 'frames': [], 'current_run': 0}); lt = None
        for i, fr in indexed_frames:
            s = fr.get('state', {}); tk = f"{s.get('x',0)}_{s.get('y',0)}"
            if tk == lt: dd[tk]['current_run'] += 1
            else:
                if lt is not None and dd[lt]['current_run'] > 0: dd[lt]['frames'].append(dd[lt]['current_run'])
                dd[tk]['visits'] += 1; dd[tk]['current_run'] = 1; lt = tk
        if lt is not None and dd[lt]['current_run'] > 0: dd[lt]['frames'].append(dd[lt]['current_run'])
        dtimes = {}
        for tk, d in dd.items():
            runs = d['frames']
            if not runs: continue
            total = sum(runs)
            dtimes[tk] = {'total_frames': total, 'visits': d['visits'], 'avg_dwell': round(total/len(runs),1), 'max_dwell': max(runs)}
        ps = []; cs = None
        for idx in range(len(indexed_frames)):
            gi, fr = indexed_frames[idx]; s = fr.get('state', {}); act = fr.get('action', 'NONE')
            x, y = s.get('x',0), s.get('y',0)
            if act not in ('UP','DOWN','LEFT','RIGHT'):
                if cs and len(cs['tiles']) >= 3:
                    cs['end'] = cs['tiles'][-1]; cs['length'] = len(cs['tiles']); cs['frame_end'] = gi; ps.append(cs)
                cs = None; continue
            if cs is None or act != cs['primary_action']:
                if cs and len(cs['tiles']) >= 3:
                    cs['end'] = cs['tiles'][-1]; cs['length'] = len(cs['tiles']); cs['frame_end'] = gi-1; ps.append(cs)
                cs = {'start': [x,y], 'end': [x,y], 'tiles': [[x,y]], 'primary_action': act, 'actions': [act], 'length': 1, 'frame_start': gi, 'frame_end': gi}
            else:
                pos = [x, y]
                if pos != cs['tiles'][-1]: cs['tiles'].append(pos)
                cs['actions'].append(act)
        if cs and len(cs['tiles']) >= 3:
            cs['end'] = cs['tiles'][-1]; cs['length'] = len(cs['tiles']); cs['frame_end'] = indexed_frames[-1][0]; ps.append(cs)
        per_map[mk] = {'action_probabilities': ap, 'movement_graph': mg_s, 'decision_points': dp, 'dwell_times': dtimes, 'path_segments': ps}
        print(f"    Map {map_id}: {len(ap)} tiles, {len(dp)} decisions, {len(ps)} paths")
    data['per_map_analysis'] = per_map
    with open(TAUGHT_TRANSITIONS_FILE, 'w') as f:
        json.dump(data, f)
    print(f"  ✅ per_map_analysis → {TAUGHT_TRANSITIONS_FILE.name}")


# =========================================================================
# POST-PROCESSOR 2: taught_nav_targets.json (UNCHANGED)
# =========================================================================
NAV_TARGETS_PATH = BASE_PATH / "taught_nav_targets.json"
ANALYSIS_WINDOW_AFTER = 40
RECENT_WINDOW = 100
MIN_FORWARD_PROGRESS = 0.5
DEDUP_RADIUS_NAV = 2
BACKTRACK_WINDOW = 50
BACKTRACK_PROXIMITY = 3

def run_nav_target_extraction():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — writing empty nav targets")
        _write_empty_nav_targets(); return
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    all_frames = []
    for batch in data.get('batches', []):
        for frame in batch.get('frames', []): all_frames.append(frame)
    if not all_frames:
        print("  ⚠️ No frames — writing empty nav targets"); _write_empty_nav_targets(); return
    print(f"  Scanning {len(all_frames)} frames for novelty...")
    novelty_points = []
    for i, frame in enumerate(all_frames):
        s = frame.get('state', {}); x, y = s.get('x',0), s.get('y',0)
        mid = s.get('map_id',0); d = s.get('direction',0)
        ib, im = s.get('in_battle',0), s.get('in_menu',0)
        act = frame.get('action', 'NONE')
        ps = all_frames[i-1].get('state', {}) if i > 0 else {}
        pmid, pib = ps.get('map_id', mid), ps.get('in_battle', 0)
        if i > 0 and mid != pmid:
            px, py = ps.get('x', x), ps.get('y', y)
            novelty_points.append({'position': [px,py], 'map_id': pmid, 'direction': ps.get('direction',d),
                'frame_index': i, 'novelty_type': 'map_transition', 'destination_map': mid}); continue
        if ib == 1 and pib == 0:
            novelty_points.append({'position': [x,y], 'map_id': mid, 'direction': d,
                'frame_index': i, 'novelty_type': 'battle', 'destination_map': None}); continue
        if act == 'A' and ib == 0 and im == 0:
            triggered = False
            for j in range(i+1, min(i+9, len(all_frames))):
                fs = all_frames[j].get('state', {})
                if fs.get('in_menu', 0) != im or fs.get('map_id', mid) != mid: triggered = True; break
            if triggered:
                novelty_points.append({'position': [x,y], 'map_id': mid, 'direction': d,
                    'frame_index': i, 'novelty_type': 'interaction', 'destination_map': None}); continue
        if ib == 0 and im == 0 and i > RECENT_WINDOW:
            was_recent = any(all_frames[j].get('state',{}).get('map_id')==mid and all_frames[j].get('state',{}).get('x')==x and all_frames[j].get('state',{}).get('y')==y for j in range(max(0,i-RECENT_WINDOW), max(0,i-5)))
            if not was_recent:
                too_close = novelty_points and novelty_points[-1]['map_id']==mid and abs(novelty_points[-1]['position'][0]-x)+abs(novelty_points[-1]['position'][1]-y)<=DEDUP_RADIUS_NAV
                if not too_close:
                    novelty_points.append({'position': [x,y], 'map_id': mid, 'direction': d,
                        'frame_index': i, 'novelty_type': 'new_area', 'destination_map': None})
    tc = defaultdict(int)
    for np_item in novelty_points: tc[np_item['novelty_type']] += 1
    print(f"    Novelty points: {len(novelty_points)} ({', '.join(f'{t}:{c}' for t,c in tc.items())})")
    scored = []
    for np_item in novelty_points:
        fi = np_item['frame_index']; mid = np_item['map_id']; px, py = np_item['position']
        before = set()
        for j in range(max(0,fi-RECENT_WINDOW), fi):
            js = all_frames[j].get('state', {})
            if js.get('in_battle',0)==0 and js.get('in_menu',0)==0:
                before.add((js.get('map_id',0), js.get('x',0), js.get('y',0)))
        after_new, after_total = 0, 0
        for j in range(fi+1, min(fi+1+ANALYSIS_WINDOW_AFTER, len(all_frames))):
            js = all_frames[j].get('state', {})
            if js.get('in_battle',0)==1 or js.get('in_menu',0)==1: continue
            after_total += 1
            if (js.get('map_id',0), js.get('x',0), js.get('y',0)) not in before: after_new += 1
        fwd = after_new / after_total if after_total > 0 else 0.0
        bt = False
        if np_item['novelty_type'] == 'map_transition':
            for j in range(fi+1, min(fi+1+BACKTRACK_WINDOW, len(all_frames))):
                if all_frames[j].get('state',{}).get('map_id') == mid: bt = True; break
        else:
            for j in range(fi+5, min(fi+1+BACKTRACK_WINDOW, len(all_frames))):
                js = all_frames[j].get('state', {})
                if js.get('map_id')==mid and abs(js.get('x',0)-px)+abs(js.get('y',0)-py)<=BACKTRACK_PROXIMITY: bt = True; break
        if bt or fwd < MIN_FORWARD_PROGRESS: continue
        np_item['forward_progress_score'] = round(fwd, 3); scored.append(np_item)
    print(f"    After filtering: {len(scored)} targets")
    deduped = []
    by_map = defaultdict(list)
    for t in scored: by_map[t['map_id']].append(t)
    for mid, targets in by_map.items():
        targets.sort(key=lambda t: t['forward_progress_score'], reverse=True)
        kept = []
        for t in targets:
            tx, ty = t['position']
            if not any(abs(tx-k['position'][0])+abs(ty-k['position'][1])<=DEDUP_RADIUS_NAV for k in kept): kept.append(t)
        deduped.extend(kept)
    deduped.sort(key=lambda t: t['frame_index'])
    tbm = defaultdict(list); go = []
    for order, t in enumerate(deduped, 1):
        mk = str(t['map_id'])
        tbm[mk].append({'position': t['position'], 'direction': t['direction'], 'order': order,
            'progress_type': t['novelty_type'], 'forward_progress_score': t['forward_progress_score'],
            'destination_map': t.get('destination_map'), 'frame_index': t['frame_index']})
        go.append({'map_id': t['map_id'], 'position': t['position'], 'order': order})
    output = {'targets_by_map': dict(tbm), 'global_order': go,
        'metadata': {'total_targets': len(deduped), 'maps_with_targets': sorted(set(t['map_id'] for t in deduped)),
            'analysis_window_after': ANALYSIS_WINDOW_AFTER, 'min_forward_progress': MIN_FORWARD_PROGRESS,
            'dedup_radius': DEDUP_RADIUS_NAV, 'generated_from_frames': len(all_frames)}}
    with open(NAV_TARGETS_PATH, 'w') as f:
        json.dump(output, f, indent=2)
    print(f"  ✅ taught_nav_targets.json → {len(deduped)} targets across {len(tbm)} maps")

def _write_empty_nav_targets():
    with open(NAV_TARGETS_PATH, 'w') as f:
        json.dump({'targets_by_map': {}, 'global_order': [], 'metadata': {'total_targets': 0, 'maps_with_targets': [],
            'analysis_window_after': ANALYSIS_WINDOW_AFTER, 'min_forward_progress': MIN_FORWARD_PROGRESS,
            'dedup_radius': DEDUP_RADIUS_NAV, 'generated_from_frames': 0}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 3: taught_battle_transitions.json (UPDATED)
# Now uses expanded battle_data fields from brain buffer
# =========================================================================
BATTLE_TRANSITIONS_PATH = BASE_PATH / "taught_battle_transitions.json"
POKEMON_CENTER_MAPS = {1, 2, 3, 4}

def run_battle_extraction():
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — writing empty battle transitions")
        _write_empty_battle_transitions(); return
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    all_frames = []
    for batch in data.get('batches', []):
        bt = batch.get('batch_type', 'steady')
        for frame in batch.get('frames', []):
            frame['_batch_type'] = bt
            all_frames.append(frame)
    if not all_frames:
        print("  ⚠️ No frames — writing empty battle transitions")
        _write_empty_battle_transitions(); return
    print(f"  Scanning {len(all_frames)} frames for battles...")

    battle_buffer = brain.get_all_battle_data_buffer()
    buffer_by_battle_id = defaultdict(list)
    for entry in battle_buffer:
        buffer_by_battle_id[entry['battle_id']].append(entry)
    print(f"    Battle data buffer: {len(battle_buffer)} entries across {len(buffer_by_battle_id)} battles")

    battle_sequences = []
    current_battle = None
    battle_id = 0
    for i, frame in enumerate(all_frames):
        s = frame.get('state', {})
        ib = s.get('in_battle', 0)
        if ib == 1 and current_battle is None:
            battle_id += 1
            current_battle = {'battle_id': battle_id, 'start_frame': i, 'end_frame': i,
                'map_id': s.get('map_id', 0), 'frames': [], 'recent_actions_buffer': [],
                'frame_index_in_battle': 0}
        if ib == 1 and current_battle is not None:
            action = frame.get('action', 'NONE')
            recent = frame.get('recent_actions', [])
            if len(recent) < 8: recent = (['NONE'] * (8 - len(recent))) + recent
            elif len(recent) > 8: recent = recent[-8:]
            bd_short = _lookup_battle_data(buffer_by_battle_id, battle_id, current_battle['frame_index_in_battle'])
            battle_frame = {
                'state': {'map_id': s.get('map_id', 0), 'x': s.get('x', 0), 'y': s.get('y', 0),
                    'direction': s.get('direction', 0), 'in_battle': 1, 'in_menu': s.get('in_menu', 0)},
                'action': action if action else 'NONE',
                'recent_actions': recent,
                'frame_offset': frame.get('frame_offset', 0),
                'batch_type': frame.get('_batch_type', 'steady'),
                'battle_data': bd_short}
            current_battle['frames'].append(battle_frame)
            current_battle['end_frame'] = i
            current_battle['frame_index_in_battle'] += 1
            if action and action != 'NONE': current_battle['recent_actions_buffer'].append(action)
        elif ib == 0 and current_battle is not None:
            if len(current_battle['frames']) >= 2:
                outcome = _detect_battle_outcome(current_battle, all_frames, i)
                battle_sequences.append({'battle_id': current_battle['battle_id'],
                    'start_frame': current_battle['start_frame'], 'end_frame': current_battle['end_frame'],
                    'map_id': current_battle['map_id'], 'duration_frames': len(current_battle['frames']),
                    'outcome': outcome, 'frames': current_battle['frames']})
            current_battle = None
    if current_battle is not None and len(current_battle['frames']) >= 2:
        battle_sequences.append({'battle_id': current_battle['battle_id'],
            'start_frame': current_battle['start_frame'], 'end_frame': current_battle['end_frame'],
            'map_id': current_battle['map_id'], 'duration_frames': len(current_battle['frames']),
            'outcome': 'unknown', 'frames': current_battle['frames']})

    flat_frames = []
    for seq in battle_sequences:
        for frame in seq['frames']:
            flat_frame = dict(frame); flat_frame['battle_id'] = seq['battle_id']; flat_frames.append(flat_frame)

    outcomes = defaultdict(int); maps_with_battles = set()
    for seq in battle_sequences: outcomes[seq['outcome']] += 1; maps_with_battles.add(seq['map_id'])
    avg_length = (sum(s['duration_frames'] for s in battle_sequences) / len(battle_sequences)) if battle_sequences else 0
    frames_with_bd = sum(1 for f in flat_frames if f.get('battle_data') and f['battle_data'].get('bc', -1) != -1)

    seq_counts = defaultdict(int)
    for seq in battle_sequences:
        actions = [f['action'] for f in seq['frames'] if f['action'] != 'NONE']
        for win_size in [2, 3, 4]:
            for j in range(len(actions) - win_size + 1):
                seq_counts[tuple(actions[j:j+win_size])] += 1
    common_seqs = sorted(seq_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    most_common = [{'sequence': list(s), 'count': c, 'context': _guess_sequence_context(list(s))} for s, c in common_seqs]

    metadata = {'total_battle_frames': len(flat_frames), 'battles_recorded': len(battle_sequences),
        'avg_battle_length': round(avg_length, 1), 'outcomes': dict(outcomes),
        'maps_with_battles': sorted(maps_with_battles), 'most_common_sequences': most_common,
        'frames_with_battle_data': frames_with_bd,
        'battle_data_coverage': round(frames_with_bd / max(1, len(flat_frames)), 3)}

    with open(BATTLE_TRANSITIONS_PATH, 'w') as f:
        json.dump({'battle_sequences': battle_sequences, 'flat_frames': flat_frames, 'metadata': metadata}, f)
    print(f"  ✅ taught_battle_transitions.json:")
    print(f"     Battles: {len(battle_sequences)} | Frames: {len(flat_frames)}")
    print(f"     Outcomes: {dict(outcomes)} | Avg length: {avg_length:.1f}")
    print(f"     Battle data coverage: {frames_with_bd}/{len(flat_frames)} ({metadata['battle_data_coverage']:.0%})")


def _lookup_battle_data(buffer_by_battle_id, battle_id, frame_index):
    entries = buffer_by_battle_id.get(battle_id, [])
    if not entries:
        return {'bc': -1, 'mc': -1, 'ps': -1, 'es': -1, 'ph': -1, 'pm': -1, 'eh': -1, 'em': -1,
                'pl': -1, 'el': -1, 'pst': 0, 'est': 0, 'bt': 0, 'pc': -1}
    idx = min(frame_index, len(entries) - 1)
    bd = entries[idx].get('battle_data', {})
    return {
        'bc': bd.get('battle_cursor', -1), 'mc': bd.get('move_cursor', -1),
        'ps': bd.get('player_species', -1), 'es': bd.get('enemy_species', -1),
        'ph': bd.get('player_hp', -1), 'pm': bd.get('player_max_hp', -1),
        'eh': bd.get('enemy_hp', -1), 'em': bd.get('enemy_max_hp', -1),
        'pl': bd.get('player_level', -1), 'el': bd.get('enemy_level', -1),
        'pst': bd.get('player_status', 0), 'est': bd.get('enemy_status', 0),
        'bt': bd.get('battle_type', 0), 'pc': bd.get('party_cursor', -1),
    }


def _detect_battle_outcome(battle, all_frames, end_index):
    frames = battle.get('frames', [])
    if frames:
        last_bd = frames[-1].get('battle_data', {})
        eh = last_bd.get('eh', -1); ph = last_bd.get('ph', -1)
        if eh == 0 and eh != -1: return 'win'
        if ph == 0 and ph != -1: return 'loss'
        if eh > 0 and ph > 0: pass  # Check run patterns below
    actions = battle.get('recent_actions_buffer', [])
    if len(actions) >= 3:
        last_actions = actions[-6:] if len(actions) >= 6 else actions
        for pattern in [['DOWN', 'RIGHT', 'A'], ['DOWN', 'A'], ['RIGHT', 'DOWN', 'A']]:
            plen = len(pattern)
            for k in range(len(last_actions) - plen + 1):
                if last_actions[k:k+plen] == pattern and k >= len(last_actions) - plen - 2:
                    return 'run'
    if end_index + 5 < len(all_frames):
        for j in range(end_index, min(end_index + 10, len(all_frames))):
            post_map = all_frames[j].get('state', {}).get('map_id', -1)
            if post_map in POKEMON_CENTER_MAPS and post_map != battle.get('map_id', -1):
                return 'loss'
    return 'win'


def _guess_sequence_context(seq):
    if seq == ['A', 'A', 'A', 'A']: return 'mashing_A_through_text_or_selecting_fight'
    if seq == ['A', 'A']: return 'selecting_fight_and_move'
    if 'DOWN' in seq and 'A' in seq: return 'navigating_menu_then_selecting'
    if seq == ['B', 'B']: return 'cancelling_or_backing_out'
    if all(a == 'A' for a in seq): return 'mashing_A'
    return 'battle_input_sequence'


def _write_empty_battle_transitions():
    with open(BATTLE_TRANSITIONS_PATH, 'w') as f:
        json.dump({'battle_sequences': [], 'flat_frames': [], 'metadata': {
            'total_battle_frames': 0, 'battles_recorded': 0, 'avg_battle_length': 0,
            'outcomes': {}, 'maps_with_battles': [], 'most_common_sequences': [],
            'frames_with_battle_data': 0, 'battle_data_coverage': 0.0}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 4: taught_bag_transitions.json (NEW)
# Writes accumulated bag session data from brain's buffer
# =========================================================================

def run_bag_extraction():
    """
    Write taught_bag_transitions.json from brain's accumulated bag session data.
    
    This is simpler than the other post-processors because bag frames are
    recorded LIVE during the teaching session (by brain.record_bag_frame()),
    not extracted from taught_transitions.json after the fact.
    
    The post-processor just writes what's already accumulated.
    """
    bag_data = brain.get_accumulated_bag_data()
    n_frames = len(bag_data['bag_frames'])
    n_sessions = bag_data['metadata']['bag_sessions_recorded']
    
    if n_frames == 0:
        print("  ⚠️ No bag frames recorded — writing empty bag transitions")
        _write_empty_bag_transitions()
        return
    
    with open(TAUGHT_BAG_TRANSITIONS_FILE, 'w') as f:
        json.dump(bag_data, f)
    
    items_used = bag_data['metadata']['items_used']
    pockets = bag_data['metadata']['pockets_visited']
    pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
    pk_strs = [pocket_names.get(p, f"?{p}") for p in pockets]
    
    print(f"  ✅ taught_bag_transitions.json:")
    print(f"     Frames: {n_frames} | Sessions: {n_sessions}")
    print(f"     Items used: {items_used}")
    print(f"     Pockets visited: {', '.join(pk_strs)}")


def _write_empty_bag_transitions():
    with open(TAUGHT_BAG_TRANSITIONS_FILE, 'w') as f:
        json.dump({'bag_frames': [], 'metadata': {
            'total_bag_frames': 0, 'bag_sessions_recorded': 0,
            'items_used': [], 'pockets_visited': []}}, f, indent=2)


# =========================================================================
# POST-PROCESSOR 5: event_timeline.json (NEW)
# Analyzes all recorded data to produce structured playthrough history
# =========================================================================

def run_event_timeline_generation():
    """
    Generate event_timeline.json by post-processing demonstration data.
    
    Walks through taught_transitions.json frames chronologically to detect:
    - Battle start/end (battle_flag transitions)
    - Bag sessions (game_state transitions to/from 14)
    - Map transitions (map_id changes)
    - Party switches (stat fingerprint changes)
    
    Then computes segments and preparation points.
    """
    if not TAUGHT_TRANSITIONS_FILE.exists():
        print("  ⚠️ No taught_transitions.json — writing empty event timeline")
        _write_empty_event_timeline()
        return
    
    with open(TAUGHT_TRANSITIONS_FILE, 'r') as f:
        data = json.load(f)
    
    all_frames = []
    for batch in data.get('batches', []):
        for frame in batch.get('frames', []):
            all_frames.append(frame)
    
    if not all_frames:
        print("  ⚠️ No frames — writing empty event timeline")
        _write_empty_event_timeline()
        return
    
    # Load nav targets for linking events to journey progress
    nav_targets = []
    if NAV_TARGETS_PATH.exists():
        try:
            with open(NAV_TARGETS_PATH, 'r') as f:
                nav_data = json.load(f)
            nav_targets = nav_data.get('global_order', [])
        except:
            pass
    
    print(f"  Generating event timeline from {len(all_frames)} frames...")
    print(f"    Nav targets available: {len(nav_targets)}")
    
    # === STEP 1: Walk through frames, detect event boundaries ===
    events = []
    event_order = 0
    
    # State tracking
    prev_battle = 0
    prev_map = None
    prev_gs = 0
    battle_start_frame = -1
    battle_start_party = None
    battle_enemy_species = -1
    battle_enemy_level = -1
    battle_type = 0
    battle_moves_used = set()
    battle_turn_count = 0
    
    bag_start_frame = -1
    bag_start_party = None
    bag_items_used = []
    bag_pockets_visited = set()
    
    # Party fingerprint tracking for switch detection
    prev_party_fingerprint = None
    
    for i, frame in enumerate(all_frames):
        s = frame.get('state', {})
        x, y = s.get('x', 0), s.get('y', 0)
        map_id = s.get('map_id', 0)
        battle_flag = s.get('in_battle', 0)
        gs = s.get('game_state', 0)
        direction = s.get('direction', 0)
        
        # Get party snapshot from frame state if available
        party_snapshot = _extract_party_from_frame(frame)
        
        # --- BATTLE DETECTION ---
        if battle_flag == 1 and prev_battle == 0:
            # Battle started
            battle_start_frame = i
            battle_start_party = party_snapshot
            battle_moves_used = set()
            battle_turn_count = 0
            # Try to get enemy info from battle_data in buffer
            bd_info = _get_battle_info_near_frame(i, all_frames)
            battle_enemy_species = bd_info.get('enemy_species', -1)
            battle_enemy_level = bd_info.get('enemy_level', -1)
            battle_type = bd_info.get('battle_type', 0)
        
        elif battle_flag == 0 and prev_battle == 1:
            # Battle ended
            party_after = party_snapshot
            
            # Detect outcome
            outcome = _detect_timeline_battle_outcome(all_frames, battle_start_frame, i)
            
            # Compute HP cost
            hp_cost = 0.0
            if battle_start_party and party_after:
                start_ratios = battle_start_party.get('hp_ratios', [])
                end_ratios = party_after.get('hp_ratios', [])
                if start_ratios and end_ratios:
                    costs = []
                    for sr, er in zip(start_ratios, end_ratios):
                        if sr > 0:
                            costs.append(max(0.0, sr - er))
                    hp_cost = sum(costs) / max(1, len(costs))
            
            bt_str = "trainer" if (battle_type & 8) != 0 else "wild"
            
            nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
            
            event = {
                'order': event_order,
                'type': 'battle',
                'nav_target_order': nav_order,
                'map_id': map_id,
                'position': [x, y],
                'timestamp_range': [battle_start_frame, i],
                'battle_info': {
                    'battle_type': bt_str,
                    'enemy_species': battle_enemy_species,
                    'enemy_level': battle_enemy_level,
                    'outcome': outcome,
                    'turns': battle_turn_count,
                    'hp_cost': round(hp_cost, 3),
                    'moves_used': sorted(list(battle_moves_used)),
                },
                'party_before': battle_start_party or _empty_party_snapshot(),
                'party_after': party_after or _empty_party_snapshot(),
            }
            events.append(event)
            event_order += 1
            battle_start_frame = -1
        
        # --- BAG SESSION DETECTION ---
        if gs == 14 and prev_gs != 14 and battle_flag == 0:
            # Bag opened (overworld)
            bag_start_frame = i
            bag_start_party = party_snapshot
            bag_items_used = []
            bag_pockets_visited = set()
        
        elif gs != 14 and prev_gs == 14 and battle_flag == 0:
            # Bag closed
            if bag_start_frame >= 0:
                party_after = party_snapshot
                
                # Infer reason
                reason = _infer_bag_reason(bag_start_party, party_after, bag_items_used, 
                                           events, nav_targets, x, y, map_id)
                
                nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
                
                event = {
                    'order': event_order,
                    'type': 'bag_session',
                    'nav_target_order': nav_order,
                    'map_id': map_id,
                    'position': [x, y],
                    'timestamp_range': [bag_start_frame, i],
                    'bag_info': {
                        'items_used': [{'item_id': iid, 'target_slot': -1, 'effect': 'unknown', 'hp_restored': 0}
                                       for iid in bag_items_used],
                        'pockets_visited': sorted(list(bag_pockets_visited)),
                        'reason_inferred': reason,
                    },
                    'party_before': bag_start_party or _empty_party_snapshot(),
                    'party_after': party_after or _empty_party_snapshot(),
                }
                events.append(event)
                event_order += 1
                bag_start_frame = -1
        
        # Track bag pocket visits and item uses from accumulated bag data
        if gs == 14 and bag_start_frame >= 0:
            pocket = s.get('bag_pocket', -1)
            if pocket >= 0:
                bag_pockets_visited.add(pocket)
        
        # --- MAP TRANSITION DETECTION ---
        if prev_map is not None and map_id != prev_map:
            nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
            
            event = {
                'order': event_order,
                'type': 'map_transition',
                'nav_target_order': nav_order,
                'map_id': prev_map,
                'position': [x, y],
                'destination_map': map_id,
                'timestamp_range': [max(0, i-1), i],
                'party_snapshot': party_snapshot or _empty_party_snapshot(),
            }
            events.append(event)
            event_order += 1
        
        # --- PARTY SWITCH DETECTION ---
        current_fingerprint = _get_party_fingerprint(party_snapshot)
        if (prev_party_fingerprint is not None and current_fingerprint is not None and
            current_fingerprint != prev_party_fingerprint and battle_flag == 0):
            # Party order changed outside battle
            from_slot, to_slot = _detect_switch_slots(prev_party_fingerprint, current_fingerprint)
            reason = _infer_switch_reason(party_snapshot)
            
            nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
            
            event = {
                'order': event_order,
                'type': 'party_switch',
                'nav_target_order': nav_order,
                'map_id': map_id,
                'position': [x, y],
                'timestamp_range': [max(0, i-1), i],
                'switch_info': {
                    'from_slot': from_slot,
                    'to_slot': to_slot,
                    'reason_inferred': reason,
                },
                'party_snapshot': party_snapshot or _empty_party_snapshot(),
            }
            events.append(event)
            event_order += 1
        
        if current_fingerprint is not None:
            prev_party_fingerprint = current_fingerprint
        
        prev_battle = battle_flag
        prev_map = map_id
        prev_gs = gs
    
    # === STEP 2: Add travel events between other events ===
    # (simplified — mark significant position changes as travel)
    
    # === STEP 3: Compute segments ===
    segments = _compute_segments(events, nav_targets)
    
    # === STEP 4: Identify preparation points ===
    preparation_points = _identify_preparation_points(events)
    
    # === STEP 5: Compute metadata ===
    type_counts = defaultdict(int)
    for e in events:
        type_counts[e['type']] += 1
    
    nav_orders_covered = sorted(set(e.get('nav_target_order', -1) for e in events if e.get('nav_target_order', -1) >= 0))
    
    metadata = {
        'total_events': len(events),
        'total_battles': type_counts.get('battle', 0),
        'total_bag_sessions': type_counts.get('bag_session', 0),
        'total_switches': type_counts.get('party_switch', 0),
        'total_map_transitions': type_counts.get('map_transition', 0),
        'playthrough_timesteps': len(all_frames),
        'nav_targets_covered': nav_orders_covered,
        'generation_timestamp': datetime.now().isoformat(),
    }
    
    # === STEP 6: Write output ===
    output = {
        'events': events,
        'segments': segments,
        'preparation_points': preparation_points,
        'metadata': metadata,
    }
    
    with open(EVENT_TIMELINE_FILE, 'w') as f:
        json.dump(output, f, indent=2)
    
    print(f"  ✅ event_timeline.json:")
    print(f"     Events: {len(events)} | Segments: {len(segments)} | Prep points: {len(preparation_points)}")
    print(f"     Types: {dict(type_counts)}")
    print(f"     Nav targets covered: {nav_orders_covered}")
    if preparation_points:
        for pp in preparation_points[:3]:
            print(f"     Prep: before #{pp['before_nav_order']} — {pp['reason']} (HP<{pp['party_hp_threshold']})")


# === Event timeline helper functions ===

def _extract_party_from_frame(frame):
    """Extract party snapshot from a transition frame's expanded state."""
    s = frame.get('state', {})
    # The Lua v3 teaching script stores party data in game_state.json
    # but taught_transitions.json frames don't embed full party data.
    # We return a minimal snapshot based on what's available.
    # Full party snapshots come from the brain's live party_data during recording.
    return None  # Will be populated from brain's live data during recording


def _get_battle_info_near_frame(frame_idx, all_frames):
    """Try to extract battle info from nearby frames."""
    # Battle data comes from the brain's buffer, not from transition frames
    # Return defaults — the battle extraction post-processor handles this
    return {'enemy_species': -1, 'enemy_level': -1, 'battle_type': 0}


def _detect_timeline_battle_outcome(all_frames, start_idx, end_idx):
    """Detect battle outcome from frame sequence."""
    # Check if player ended up in Pokemon Center after
    if end_idx + 10 < len(all_frames):
        for j in range(end_idx, min(end_idx + 10, len(all_frames))):
            post_map = all_frames[j].get('state', {}).get('map_id', -1)
            if post_map in POKEMON_CENTER_MAPS:
                battle_map = all_frames[start_idx].get('state', {}).get('map_id', -1)
                if post_map != battle_map:
                    return 'loss'
    
    # Check action patterns for running
    run_frames = all_frames[max(0, end_idx-10):end_idx]
    actions = [f.get('action', 'NONE') for f in run_frames if f.get('state', {}).get('in_battle', 0) == 1]
    for pattern in [['DOWN', 'RIGHT', 'A'], ['DOWN', 'A']]:
        plen = len(pattern)
        for k in range(len(actions) - plen + 1):
            if actions[k:k+plen] == pattern:
                return 'run'
    
    return 'win'


def _find_nearest_nav_order(x, y, map_id, nav_targets):
    """Find the nearest nav target order for a position."""
    if not nav_targets:
        return -1
    
    best_order = -1
    best_dist = float('inf')
    
    for t in nav_targets:
        t_map = t.get('map_id', -1)
        t_pos = t.get('position', [0, 0])
        t_order = t.get('order', -1)
        
        if t_map == map_id:
            dist = abs(x - t_pos[0]) + abs(y - t_pos[1])
            if dist < best_dist:
                best_dist = dist
                best_order = t_order
    
    # If no target on this map, find closest order from any map
    if best_order == -1 and nav_targets:
        # Use the highest order target we've passed (approximate)
        for t in reversed(nav_targets):
            best_order = t.get('order', -1)
            break
    
    return best_order


def _empty_party_snapshot():
    return {'hp_ratios': [], 'status_flags': [], 'count': 0, 'levels': []}


def _get_party_fingerprint(party_snapshot):
    """Get a hashable fingerprint of party order based on levels."""
    if party_snapshot is None:
        return None
    levels = party_snapshot.get('levels', [])
    if not levels:
        return None
    return tuple(levels)


def _detect_switch_slots(prev_fp, curr_fp):
    """Detect which slots were swapped based on fingerprint change."""
    if prev_fp is None or curr_fp is None:
        return -1, -1
    
    for i in range(min(len(prev_fp), len(curr_fp))):
        if prev_fp[i] != curr_fp[i]:
            # Find where this level went
            for j in range(len(curr_fp)):
                if j != i and curr_fp[j] == prev_fp[i]:
                    return i, j
            return i, -1
    return -1, -1


def _infer_switch_reason(party_snapshot):
    """Infer why the human switched party order."""
    if party_snapshot is None:
        return "unknown"
    
    hp_ratios = party_snapshot.get('hp_ratios', [])
    if hp_ratios and len(hp_ratios) > 0:
        if hp_ratios[0] < 0.3:
            return "lead_low_hp"
        if hp_ratios[0] < 0.5:
            return "lead_moderate_hp"
    
    status_flags = party_snapshot.get('status_flags', [])
    if status_flags and len(status_flags) > 0:
        if status_flags[0] != 0:
            return "lead_has_status"
    
    return "strategic"


def _infer_bag_reason(party_before, party_after, items_used, events, nav_targets, x, y, map_id):
    """Infer why the human opened the bag."""
    if party_before is None:
        return "unknown"
    
    hp_ratios = party_before.get('hp_ratios', [])
    lowest_hp = min((r for r in hp_ratios if r > 0), default=1.0)
    
    has_status = any(s != 0 for s in party_before.get('status_flags', []))
    
    if lowest_hp < 0.5:
        return "low_hp"
    
    if has_status:
        return "status_condition"
    
    # Check if a battle follows soon in the timeline
    upcoming_battles = [e for e in events if e.get('type') == 'battle' and 
                        e.get('nav_target_order', -1) >= 0]
    if upcoming_battles:
        nav_order = _find_nearest_nav_order(x, y, map_id, nav_targets)
        close_battles = [b for b in upcoming_battles 
                        if 0 < b.get('nav_target_order', -1) - nav_order <= 3]
        if close_battles:
            trainer_battles = [b for b in close_battles 
                              if b.get('battle_info', {}).get('battle_type') == 'trainer']
            if trainer_battles:
                return "pre_trainer_prep"
            return "pre_battle_prep"
    
    if lowest_hp >= 0.9:
        return "item_management"
    
    return "other"


def _compute_segments(events, nav_targets):
    """Compute difficulty segments between nav target orders."""
    if not events or not nav_targets:
        return []
    
    # Group events by nav target order ranges
    orders_seen = sorted(set(e.get('nav_target_order', -1) for e in events if e.get('nav_target_order', -1) >= 0))
    
    if len(orders_seen) < 2:
        return []
    
    segments = []
    for i in range(len(orders_seen) - 1):
        from_order = orders_seen[i]
        to_order = orders_seen[i + 1]
        
        # Find events in this range
        range_events = [e for e in events 
                       if from_order <= e.get('nav_target_order', -1) < to_order]
        
        battles = [e for e in range_events if e.get('type') == 'battle']
        trainer_battles = [b for b in battles if b.get('battle_info', {}).get('battle_type') == 'trainer']
        wild_battles = [b for b in battles if b.get('battle_info', {}).get('battle_type') == 'wild']
        bag_sessions = [e for e in range_events if e.get('type') == 'bag_session']
        map_transitions = [e for e in range_events if e.get('type') == 'map_transition']
        
        hp_costs = [b.get('battle_info', {}).get('hp_cost', 0.0) for b in battles]
        avg_hp_cost = sum(hp_costs) / max(1, len(hp_costs))
        total_hp_cost = sum(hp_costs)
        
        segment = {
            'from_nav_order': from_order,
            'to_nav_order': to_order,
            'battles_expected': len(battles),
            'avg_hp_cost_per_battle': round(avg_hp_cost, 3),
            'total_hp_cost': round(total_hp_cost, 3),
            'bag_sessions': len(bag_sessions),
            'trainer_battles': len(trainer_battles),
            'wild_battles': len(wild_battles),
            'map_transitions': len(map_transitions),
        }
        segments.append(segment)
    
    return segments


def _identify_preparation_points(events):
    """
    Find moments where the human prepared before a challenging section.
    A preparation point is a bag_session or party_switch that occurred
    when party HP was below full, followed by a battle.
    """
    preparation_points = []
    
    for i, event in enumerate(events):
        if event.get('type') not in ('bag_session', 'party_switch'):
            continue
        
        # Get party HP at this point
        party = event.get('party_before') or event.get('party_snapshot')
        if party is None:
            continue
        
        hp_ratios = party.get('hp_ratios', [])
        if not hp_ratios:
            continue
        
        living_ratios = [r for r in hp_ratios if r > 0]
        if not living_ratios:
            continue
        
        lowest_hp = min(living_ratios)
        avg_hp = sum(living_ratios) / len(living_ratios)
        
        # Only count as preparation if HP wasn't full
        if avg_hp >= 0.95:
            continue
        
        # Check if a battle follows within the next few events
        battle_follows = False
        battle_is_trainer = False
        for j in range(i + 1, min(i + 5, len(events))):
            if events[j].get('type') == 'battle':
                battle_follows = True
                if events[j].get('battle_info', {}).get('battle_type') == 'trainer':
                    battle_is_trainer = True
                break
        
        if not battle_follows:
            continue
        
        nav_order = event.get('nav_target_order', -1)
        
        # Determine reason
        reason = event.get('bag_info', {}).get('reason_inferred', 'unknown')
        if event.get('type') == 'party_switch':
            reason = event.get('switch_info', {}).get('reason_inferred', 'unknown')
        if battle_is_trainer:
            reason = "trainer_battle_ahead"
        elif reason == 'unknown' and lowest_hp < 0.5:
            reason = "low_hp_before_battle"
        
        action_taken = event.get('type')  # 'bag_session' or 'party_switch'
        items_used = []
        if event.get('type') == 'bag_session':
            items_used = [item.get('item_id', -1) for item in 
                         event.get('bag_info', {}).get('items_used', [])]
        
        prep_point = {
            'before_nav_order': nav_order,
            'reason': reason,
            'party_hp_threshold': round(lowest_hp, 3),
            'action_taken': action_taken,
            'items_used': items_used,
        }
        preparation_points.append(prep_point)
    
    return preparation_points


def _write_empty_event_timeline():
    with open(EVENT_TIMELINE_FILE, 'w') as f:
        json.dump({
            'events': [], 'segments': [], 'preparation_points': [],
            'metadata': {
                'total_events': 0, 'total_battles': 0, 'total_bag_sessions': 0,
                'total_switches': 0, 'total_map_transitions': 0,
                'playthrough_timesteps': 0, 'nav_targets_covered': [],
                'generation_timestamp': datetime.now().isoformat(),
            }
        }, f, indent=2)


# =========================================================================
# MASTER POST-PROCESSING RUNNER
# =========================================================================

def run_all_post_processing():
    """Run all 5 post-processors."""
    print("\n  📊 Running per_map_analysis...")
    try: run_per_map_analysis()
    except Exception as e: print(f"    ⚠️ per_map_analysis failed: {e}")
    
    print("  📊 Running nav target extraction...")
    try: run_nav_target_extraction()
    except Exception as e: print(f"    ⚠️ Nav targets failed: {e}")
    
    print("  📊 Running battle extraction...")
    try: run_battle_extraction()
    except Exception as e: print(f"    ⚠️ Battle extraction failed: {e}")
    
    print("  📊 Running bag extraction...")
    try: run_bag_extraction()
    except Exception as e: print(f"    ⚠️ Bag extraction failed: {e}")
    
    print("  📊 Running event timeline generation...")
    try: run_event_timeline_generation()
    except Exception as e: print(f"    ⚠️ Event timeline failed: {e}")

In [ ]:
# ============================================================================
# CELL 6B: Teaching Mode Main Loop
# ============================================================================
# CHANGES FROM PREVIOUS:
# 1. read_game_state() unpacks 10 values (added party, gs, menu, bag)
# 2. brain.update_party_data(), update_menu_data(), update_bag_data(),
#    game_state_raw called every frame
# 3. Human action detection from Lua input_cache.txt — reads current button
#    press and feeds to brain.set_human_action() BEFORE learn()
# 4. Bag session recording: detect_bag_session_boundaries() + record_bag_frame()
#    called every frame to record bag demonstrations live
# 5. prev_game_state_raw tracked for bag session boundary detection
# 6. Updated logging: shows gs, party HP, bag session status, entity stats
# 7. Updated shutdown: reports bag sessions, entity counts, all 7 output files
# 8. run_all_post_processing() now calls all 5 post-processors
# ============================================================================

# =========================================================================
# HUMAN ACTION READER
# =========================================================================

def read_human_action_from_state():
    """
    Read the human's current button press.
    
    The Lua teaching script calls joypad.get() every frame and records
    the human's input. We can detect it from the input_cache.txt file
    which contains lines of: action,x,y,map,battle,menu,direction
    
    We read the LAST line to get the most recent human action.
    Returns expanded action name (UP/DOWN/LEFT/RIGHT/A/B/Start/Select) or None.
    """
    try:
        if not INPUT_FILE.exists():
            return None
        with open(INPUT_FILE, 'r') as f:
            content = f.read().strip()
        if not content:
            return None
        # Get last line
        lines = content.split('\n')
        last_line = lines[-1].strip()
        if not last_line:
            return None
        # Format: action,x,y,map,battle,menu,direction
        parts = last_line.split(',')
        if parts:
            short_action = parts[0].strip()
            # Expand short codes
            return ACTION_MAP.get(short_action, short_action)
    except:
        return None
    return None


def read_human_action_from_transitions():
    """
    Alternative: read human action from the Lua transition batches.
    The Lua script records the human's action in each frame record.
    We check if a new frame was added since last check.
    
    This is more reliable than input_cache since it's synchronized
    with the frame recording, but requires the transitions file to be
    freshly written.
    """
    # For now, we use the input_cache approach which is simpler
    # and doesn't require re-reading the transitions file every frame
    return read_human_action_from_state()


# =========================================================================
# BRAIN INITIALIZATION
# =========================================================================

brain = Brain()

for b in ["UP", "DOWN", "LEFT", "RIGHT"]:
    brain.add(Perceptron("action", action=b, group="move"))
for b in ["A", "B", "Start", "Select"]:
    brain.add(Perceptron("action", action=b, group="interact"))

TAUGHT_MODEL_SAVE = BASE_PATH / "taught_model_checkpoint.json"
TAUGHT_EXPLORATION_SAVE = BASE_PATH / "taught_exploration_memory.json"
brain.EXPLORATION_MEMORY_FILE = TAUGHT_EXPLORATION_SAVE

# Resume if existing
if TAUGHT_MODEL_SAVE.exists():
    loaded_ts = brain.load_taught_model(TAUGHT_MODEL_SAVE)
    print(f"🎓 RESUME: Loaded taught model from timestep {loaded_ts}")
    print(f"   Utilities: {[f'{a.action}:{a.utility:.3f}' for a in brain.actions()]}")
    print(f"   Entities: {len(brain.entities())} ({brain.entity_spawn_count} spawned, {brain.entity_merge_count} merged)")
else:
    print("🎓 FRESH START: No existing taught model")

if TAUGHT_EXPLORATION_SAVE.exists():
    brain.load_exploration_memory()
    print(f"   Exploration: {len(brain.exploration_memory)} maps loaded")

# Load existing bag data if resuming
if TAUGHT_BAG_TRANSITIONS_FILE.exists():
    try:
        with open(TAUGHT_BAG_TRANSITIONS_FILE, 'r') as f:
            existing_bag = json.load(f)
        existing_frames = existing_bag.get('bag_frames', [])
        existing_meta = existing_bag.get('metadata', {})
        if existing_frames:
            brain.bag_sessions_accumulated = existing_frames
            brain.bag_sessions_metadata['total_bag_frames'] = existing_meta.get('total_bag_frames', len(existing_frames))
            brain.bag_sessions_metadata['bag_sessions_recorded'] = existing_meta.get('bag_sessions_recorded', 0)
            brain.bag_sessions_metadata['items_used'] = set(existing_meta.get('items_used', []))
            brain.bag_sessions_metadata['pockets_visited'] = set(existing_meta.get('pockets_visited', []))
            print(f"   Bag data: {len(existing_frames)} frames from {existing_meta.get('bag_sessions_recorded', 0)} sessions resumed")
    except Exception as e:
        print(f"   ⚠️ Error loading existing bag data: {e}")

prev_context_state = None
prev_raw_position = None
last_human_action = None

print("="*70)
print("TEACHING MODE v3 — Human plays, Brain learns")
print("="*70)
print(f"  Model → {TAUGHT_MODEL_SAVE.name}")
print(f"  Exploration → {TAUGHT_EXPLORATION_SAVE.name}")
print(f"  Transitions → {TAUGHT_TRANSITIONS_FILE.name} (Lua)")
print(f"  Nav targets → {NAV_TARGETS_PATH.name} (auto)")
print(f"  Battle data → {BATTLE_TRANSITIONS_PATH.name} (auto)")
print(f"  Bag data → {TAUGHT_BAG_TRANSITIONS_FILE.name} (auto + live)")
print(f"  Timeline → {EVENT_TIMELINE_FILE.name} (auto)")
print("="*70)
print("NEW IN THIS VERSION:")
print(f"  - Human action tracking: brain.last_action set from Lua input")
print(f"  - Directed learning: action perceptrons learn from human choices")
print(f"  - Entity spawning: novel states create learned activation patterns")
print(f"  - Entity clustering: similar entities merged into robust detectors")
print(f"  - Bag session recording: live capture of bag navigation demos")
print(f"  - Full state parsing: gs, mu, bg, pa fields captured every frame")
print(f"  - Event timeline: post-processed playthrough history with prep points")
print("="*70)
print("Lua v3 fields captured:")
print(f"  gs (game_state): 0=OW, 1=menu/party, 14=bag")
print(f"  mu (menu): mc=cursor, mm=max, pc=party cursor, sc=swap")
print(f"  bg (bag): pk=pocket, bc=cursor, a=active, it=items[]")
print(f"  pa (party): c=count, s=[{{l,h,m,a,d,sp,sa,sd,st}}, ...]")
print(f"  b (battle): full mirror data when in battle")
print("="*70)
print("Play the game. Ctrl+C to stop, save, and post-process.")
print("="*70)

try:
    while True:
        # === READ GAME STATE (10 values) ===
        (context_state, palette_state, tile_state, dead, raw_position,
         battle_data, party_data, game_state_raw, menu_data, bag_data) = read_game_state()
        
        if np.sum(np.abs(context_state)) < 0.001:
            time.sleep(0.02); continue
        
        raw_x, raw_y = raw_position
        in_battle = context_state[3]
        current_map = int(context_state[2])
        current_dir = int(context_state[5])
        
        # === UPDATE ALL DATA EVERY FRAME ===
        brain.update_battle_data(battle_data, in_battle)
        brain.update_party_data(party_data)
        brain.update_menu_data(menu_data)
        brain.update_bag_data(bag_data)
        brain.prev_game_state_raw = brain.game_state_raw
        brain.game_state_raw = game_state_raw
        
        # === READ HUMAN ACTION ===
        human_action = read_human_action_from_state()
        if human_action and human_action != last_human_action:
            brain.set_human_action(human_action)
            last_human_action = human_action
        elif human_action:
            # Same action held — still set it for continuous tracking
            brain.last_action = human_action
            brain.human_action = human_action
        
        # === BAG SESSION RECORDING (every frame) ===
        bag_boundary = brain.detect_bag_session_boundaries()
        if bag_boundary == "opened_overworld":
            brain.start_bag_session("overworld")
        elif bag_boundary == "opened_battle":
            brain.start_bag_session("battle")
        elif bag_boundary == "closed":
            brain.end_bag_session()
        
        # Record bag frame if session is active
        if brain.bag_session_active and human_action:
            brain.record_bag_frame(human_action)
        
        # === CORE LEARNING ===
        brain.update_position(raw_x, raw_y)
        derived = compute_derived_features(context_state, prev_context_state)
        learning_state = build_learning_state(derived, palette_state, tile_state, in_battle)
        brain.log_state(learning_state, context_state)

        time.sleep(0.02)

        # Read next state for learning
        (next_ctx, next_pal, next_til, next_dead, next_raw_pos,
         next_battle_data, next_party_data, next_gs_raw,
         next_menu_data, next_bag_data) = read_game_state()
        next_derived = compute_derived_features(next_ctx, context_state)
        next_learning_state = build_learning_state(next_derived, next_pal, next_til, next_ctx[3])

        brain.learn(learning_state, next_learning_state, context_state, next_ctx,
                    dead=dead, raw_position=raw_position, next_raw_position=next_raw_pos)

        prev_context_state = context_state.copy()
        prev_raw_position = raw_position
        brain.timestep += 1

        # === LOGGING (every 100 steps) ===
        if brain.timestep % 100 == 0:
            mem = brain.get_current_map_memory(current_map)
            vc = len(mem['visited_tiles']); oc = len(mem['obstructions'])
            ic = len(mem['interactable_objects']); cov = brain.get_exploration_coverage(current_map)
            ts = brain.get_tile_interaction_stats(current_map)
            dn = brain.DIRECTION_NAMES.get(current_dir, '?')
            tr = mem.get('transitions', [])
            
            gs_names = {0: "OW", 1: "MENU", 14: "BAG"}
            gs_str = gs_names.get(game_state_raw, f"GS{game_state_raw}")
            
            print(f"\n{'='*60}")
            print(f"Step {brain.timestep} | Map {current_map} | ({raw_x},{raw_y}) {dn} | gs={gs_str} | Battle:{int(in_battle)}")
            
            # Human action info
            print(f"  Human action: {brain.human_action or 'None'} | Last: {brain.last_action or 'None'}")
            
            # Party status
            if party_data.get('count', 0) > 0:
                hp_ratios = brain.get_party_hp_ratios()
                levels = brain.get_party_levels()
                lowest = brain.get_lowest_hp_ratio()
                has_status = brain.has_status_condition_in_party()
                hp_strs = [f"{r:.0%}" for r in hp_ratios]
                print(f"  Party ({party_data['count']}): HP={','.join(hp_strs)} Lv={levels} lowest={lowest:.0%} status={'YES' if has_status else 'no'}")
            
            # Battle info
            if in_battle > 0.5:
                bd = brain.battle_data
                cursor_names = {0: 'FIGHT', 1: 'BAG', 2: 'PKMN', 3: 'RUN'}
                bc_str = cursor_names.get(bd.get('battle_cursor', -1), f"?({bd.get('battle_cursor', -1)})")
                bt = bd.get('battle_type', 0)
                bt_str = "TRAINER" if (bt & 8) != 0 else "WILD"
                print(f"  ⚔️ BATTLE #{brain.current_battle_id}: {bt_str} | Cursor:{bc_str}")
                if bd.get('player_species', -1) > 0:
                    print(f"     Player: sp{bd['player_species']} Lv{bd.get('player_level','?')} HP {bd['player_hp']}/{bd['player_max_hp']}")
                if bd.get('enemy_species', -1) > 0:
                    print(f"     Enemy:  sp{bd['enemy_species']} Lv{bd.get('enemy_level','?')} HP {bd['enemy_hp']}/{bd.get('enemy_max_hp','?')}")
            
            # Menu info
            if game_state_raw > 0 and in_battle <= 0.5:
                md = brain.menu_data
                print(f"  Menu: mc={md['mc']}/{md['mm']} pc={md['pc']} sc={md['sc']}")
            
            # Bag info
            if game_state_raw == 14 or brain.bag_session_active:
                bgd = brain.bag_data
                pocket_names = {0: "Items", 1: "KeyItems", 2: "Pokeballs", 3: "TMs", 4: "Berries"}
                pk_str = pocket_names.get(bgd['pocket'], f"?{bgd['pocket']}")
                n_items = len(bgd['items'])
                item_at_cur = brain.get_item_at_cursor()
                item_str = f" item={item_at_cur}" if item_at_cur > 0 else ""
                session_str = f" SESSION({brain.bag_session_context}, {brain.bag_session_frame_count}f)" if brain.bag_session_active else ""
                print(f"  🎒 Bag: {pk_str} cursor={bgd['cursor']} items={n_items}{item_str}{session_str}")
            
            # Bag recording stats
            bag_stats = brain.get_bag_recording_stats()
            if bag_stats['total_sessions'] > 0:
                print(f"  🎒 Recording: {bag_stats['total_sessions']} sessions, {bag_stats['total_frames']} frames")
                if bag_stats['items_used']:
                    print(f"     Items used: {bag_stats['items_used']}")
            
            # Exploration
            print(f"  Visited:{vc} Obs:{oc} Coverage:{cov:.0%} Interactables:{ic}")
            print(f"  Probed:{ts['probed']} Exhausted:{ts['exhausted']} Success:{ts['with_success']}")
            if tr: print(f"  Transitions: {len(tr)} known")
            
            # Entity stats
            n_entities = len(brain.entities())
            innate = sum(1 for e in brain.entities() if e.entity_type in {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"})
            spawned = n_entities - innate
            print(f"  🧩 Entities: {n_entities} ({innate} innate, {spawned} spawned, {brain.entity_merge_count} merged)")
            print(f"     Capacity: {brain.entity_capacity} | Spawn threshold active: {len(brain.error_history) >= 100}")
            
            # Stagnation / pattern
            if brain.state_stagnation_count > 5:
                print(f"  ⚠️ Stagnation: {brain.state_stagnation_count}/{brain.STATE_STAGNATION_THRESHOLD}")
            if brain.detected_pattern:
                pattern_str = '-'.join(str(a) for a in brain.detected_pattern)
                print(f"  🔄 Pattern: {pattern_str} x{brain.pattern_repeat_count}")
            
            # Battle buffer
            print(f"  Battle buffer: {len(brain.battle_data_buffer)} entries ({brain.current_battle_id} battles)")
            
            # Utilities
            au = sorted([(a.action, a.utility) for a in brain.actions()], key=lambda x: x[1], reverse=True)
            print(f"  Utilities: {' '.join(f'{k}:{v:.2f}' for k,v in au)}")

        # === PERIODIC SAVE (every 500 steps) ===
        if brain.timestep % 500 == 0 and brain.timestep > 0:
            brain.save_model_checkpoint(TAUGHT_MODEL_SAVE)
            brain.save_exploration_memory()
            # Also save bag data periodically
            try:
                run_bag_extraction()
            except Exception as e:
                print(f"  ⚠️ Periodic bag save failed: {e}")
            print(f"  💾 Auto-saved at step {brain.timestep}")
        
        # === PERIODIC POST-PROCESSING (every 2000 steps) ===
        if brain.timestep % 2000 == 0 and brain.timestep > 0:
            print(f"\n  📊 Periodic post-processing at step {brain.timestep}...")
            run_all_post_processing()
            print(f"  📊 Post-processing complete")

except KeyboardInterrupt:
    print("\n\n🛑 Stopping teaching...")
    
    # Close any active bag session
    if brain.bag_session_active:
        brain.end_bag_session()
    
    print("\n📁 Step 1/2: Saving model + exploration...")
    brain.save_model_checkpoint(TAUGHT_MODEL_SAVE)
    brain.save_exploration_memory()
    print(f"   ✅ {TAUGHT_MODEL_SAVE.name}")
    print(f"   ✅ {TAUGHT_EXPLORATION_SAVE.name}")
    
    print("\n📁 Step 2/2: Running all post-processors...")
    run_all_post_processing()
    
    # Summary
    total_tiles = sum(len(m['visited_tiles']) for m in brain.exploration_memory.values())
    total_entities = len(brain.entities())
    innate_count = sum(1 for e in brain.entities() if e.entity_type in {"sense_menu", "sense_battle", "sense_movement", "sense_map_transition"})
    spawned_count = total_entities - innate_count
    bag_stats = brain.get_bag_recording_stats()
    
    print(f"\n{'='*60}")
    print(f"✅ TEACHING COMPLETE")
    print(f"   Timestep: {brain.timestep}")
    print(f"   Maps: {len(brain.exploration_memory)}")
    print(f"   Tiles: {total_tiles}")
    print(f"   Battles buffered: {brain.current_battle_id}")
    print(f"   Battle data entries: {len(brain.battle_data_buffer)}")
    print(f"   Bag sessions: {bag_stats['total_sessions']}")
    print(f"   Bag frames: {bag_stats['total_frames']}")
    if bag_stats['items_used']:
        print(f"   Items used in bag: {bag_stats['items_used']}")
    print(f"   Entities: {total_entities} ({innate_count} innate, {spawned_count} spawned)")
    print(f"   Entity merges: {brain.entity_merge_count}")
    print(f"   Entity capacity: {brain.entity_capacity}")
    print(f"\n📦 Files ready to copy to AI agent:")
    print(f"   1. {TAUGHT_MODEL_SAVE.name} (model + entities)")
    print(f"   2. {TAUGHT_TRANSITIONS_FILE.name} (overworld demos)")
    print(f"   3. {TAUGHT_EXPLORATION_SAVE.name} (exploration memory)")
    print(f"   4. {NAV_TARGETS_PATH.name} (navigation waypoints)")
    print(f"   5. {BATTLE_TRANSITIONS_PATH.name} (battle demos)")
    print(f"   6. {TAUGHT_BAG_TRANSITIONS_FILE.name} (bag demos)")
    print(f"   7. {EVENT_TIMELINE_FILE.name} (playthrough timeline)")
    print(f"\n   Utilities: {[f'{a.action}:{a.utility:.3f}' for a in brain.actions()]}")
    print(f"{'='*60}")